# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 実験音韻論の方法 ── 音韻理論をデータで検証する ── 分析コード集(全章まとめ)


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

この1本には第2〜16章(コードを伴う章)の全スクリプトを、本書の章順にまとめてある。特定の章だけを試したい場合は、章ごとに分割された`ch02.ipynb`〜`ch16.ipynb`を使うほうが読みやすい。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


# 第2章 研究課題と仮説の設計

## `ch02_hypothesis_descent.R`

対象章: 第2章「研究課題と仮説の設計」

「理論的問いから検証可能な仮説へ」節のフロー図。理論的な問いが、
予測・操作的定義を経て測定可能な変数へと降下していく段階を、
本文の具体例(モーラの実在性)に沿って図示する概念図。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

stages <- tibble(
  step  = 1:4,
  label = c(
    "理論的な問い\n「モーラは心理的に実在するか」",
    "具体的な予測\n「モーラ境界での判断は、\nより速く・より一貫しているはずだ」",
    "操作的定義\n「速さ」→反応時間\n「一貫性」→正答率のばらつき",
    "測定可能な変数\n反応時間(ms)、正答率のSD"
  ),
  y = rev(1:4)
)

p <- ggplot(stages) +
  geom_rect(aes(xmin = 0, xmax = 10, ymin = y - 0.35, ymax = y + 0.35),
            fill = "#34518A", alpha = 0.12, colour = "#34518A") +
  geom_text(aes(x = 5, y = y, label = label), size = 3.2, colour = "#1B2333", lineheight = 0.9) +
  geom_segment(data = stages |> filter(step < 4),
               aes(x = 5, xend = 5, y = y - 0.38, yend = y - 0.65),
               arrow = arrow(length = unit(0.18, "cm"), type = "closed"), colour = "#A8681F") +
  scale_x_continuous(limits = c(0, 10)) +
  scale_y_continuous(limits = c(0.5, 4.5)) +
  theme_void() +
  labs(caption = "概念図。scripts/R/ch02_hypothesis_descent.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig02-01-hypothesis-descent", chapter = "ch02", width = 5.5, height = 6)
cat("完了。図はassets/figures/ch02/に保存した。\n")


## `ch02_within_between_design.R`

対象章: 第2章「研究課題と仮説の設計」

「実験デザインの基本用語」節の作例。同一の効果量を、参加者内デザイン
(個人差を対応のあるt検定で除去)と参加者間デザイン(個人差を除去せず
対応のないt検定)のそれぞれで検出する経験的検定力を、実際に
シミュレーションで比較する。架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_participants <- 20
true_effect    <- 10     # 条件差(理論的に意味のある最小の効果量)
individual_sd  <- 25     # 参加者間の個人差(ベースラインの違い)
residual_sd    <- 12     # 試行ごとの残差誤差
n_sim <- 500

simulate_within <- function() {
  baseline <- rnorm(n_participants, 200, individual_sd)
  cond_a <- baseline + rnorm(n_participants, 0, residual_sd)
  cond_b <- baseline + true_effect + rnorm(n_participants, 0, residual_sd)
  t.test(cond_b, cond_a, paired = TRUE)$p.value < .05
}

simulate_between <- function() {
  baseline_a <- rnorm(n_participants, 200, individual_sd)
  baseline_b <- rnorm(n_participants, 200, individual_sd)
  cond_a <- baseline_a + rnorm(n_participants, 0, residual_sd)
  cond_b <- baseline_b + true_effect + rnorm(n_participants, 0, residual_sd)
  t.test(cond_b, cond_a, paired = FALSE)$p.value < .05
}

power_within  <- mean(replicate(n_sim, simulate_within()))
power_between <- mean(replicate(n_sim, simulate_between()))

cat("参加者内デザイン(個人差を対応のあるt検定で除去): 検定力 =",
    round(power_within * 100, 1), "%\n")
cat("参加者間デザイン(個人差を除去せず): 検定力 =",
    round(power_between * 100, 1), "%\n")


## `ch02_pilot_inflation.R`

対象章: 第2章「研究課題と仮説の設計」

「検定力とサンプルサイズ」節の作例。小規模な予備実験(パイロット
スタディ)で「有意だった」効果量だけを採用してサンプルサイズを見積もると、
その効果量が真の効果量を系統的に過大評価してしまう現象(いわゆる
winner's curse)を、実際にシミュレーションで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_effect   <- 8
sd_pop        <- 20
n_pilot       <- 12
n_sim         <- 2000

pilot_once <- function() {
  a <- rnorm(n_pilot, 100, sd_pop)
  b <- rnorm(n_pilot, 100 + true_effect, sd_pop)
  test <- t.test(b, a)
  list(sig = test$p.value < .05, estimate = mean(b) - mean(a))
}

results <- map(1:n_sim, ~ pilot_once())
sig_results <- keep(results, ~ .x$sig)

mean_all_estimates <- mean(map_dbl(results, "estimate"))
mean_sig_estimates <- mean(map_dbl(sig_results, "estimate"))

cat("真の効果量:", true_effect, "\n")
cat("全", n_sim, "回の平均推定値(有意/非有意を問わず):", round(mean_all_estimates, 1), "\n")
cat("有意だった", length(sig_results), "回(", round(length(sig_results) / n_sim * 100, 1),
    "%)だけに限った平均推定値:", round(mean_sig_estimates, 1), "\n")
cat("過大評価の倍率:", round(mean_sig_estimates / true_effect, 2), "倍\n")


## `ch02_power_simulation.R`

対象章: 第2章「研究課題と仮説の設計」

「検定力とサンプルサイズ」節の作例。Westfall, Kenny, and Judd (2014)
が示した「話者数だけを増やしても、項目数が少なければ検定力は頭打ちになる」
という主張を、架空のデータに対する実際のシミュレーションで確認する。
実測データではない。

設計: 話者数(20名/100名)×項目数(5語/20語)の2×2で、各組み合わせにつき
5000回のシミュレーションを繰り返し、条件差を検出できた割合(検定力)を
経験的に求める。真の効果量は、話者・項目のランダム切片によるばらつきに
対して中程度になるよう設定する。反復数は、報告する検定力の値どうしを
数ポイント単位で比較できる程度までMonte Carlo標準誤差を抑えるために
5000回とした(真の検定力が15%付近ならSEは約0.5ポイント、45%付近でも
約0.7ポイントで、150回時点のSE(4ポイント前後)より1桁小さい)。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_effect     <- 14    # 条件差の真の値(ms、小さめに設定し天井効果を避ける)
speaker_sd      <- 20    # 話者間のランダム切片のSD
item_sd         <- 25    # 項目間のランダム切片のSD(項目数が少ないと効いてくる)
residual_sd     <- 25    # 試行内の残差SD
n_reps_per_cell <- 3     # 話者×項目×条件あたりの反復数
n_sim           <- 5000  # 各組み合わせでのシミュレーション回数

simulate_one <- function(n_speakers, n_items) {
  # 各項目は条件A・条件Bのどちらか一方にのみ属する(語彙リストA/Bのように、
  # 刺激そのものが条件を担う設計)。これにより、項目数が少ないと項目間の
  # ばらつきが条件効果の推定と直接交絡する、Clark (1973)の
  # "language-as-fixed-effect-fallacy"型の状況を再現できる。
  speakers <- tibble(speaker = sprintf("s%02d", 1:n_speakers), spk_re = rnorm(n_speakers, 0, speaker_sd))
  items <- tibble(
    item      = sprintf("i%02d", 1:(2 * n_items)),
    condition = rep(c("A", "B"), each = n_items),
    item_re   = rnorm(2 * n_items, 0, item_sd)
  )
  d <- speakers |>
    tidyr::crossing(items) |>
    tidyr::crossing(rep = 1:n_reps_per_cell) |>
    mutate(
      cond_c = if_else(condition == "B", 0.5, -0.5),
      y = 200 + true_effect * cond_c + spk_re + item_re + rnorm(n(), 0, residual_sd)
    )
  m <- tryCatch(
    lmerTest::lmer(y ~ condition + (1 | speaker) + (1 | item), data = d,
                    control = lmerControl(optimizer = "bobyqa")),
    error = function(e) NULL, warning = function(w) NULL
  )
  if (is.null(m)) return(NA)
  p <- tryCatch(summary(m)$coefficients["conditionB", "Pr(>|t|)"], error = function(e) NA)
  p
}

conditions_grid <- tibble::tribble(
  ~n_speakers, ~n_items,
  20,          5,
  100,         5,
  20,          20,
  100,         20
)

set.seed(20260906)
results <- conditions_grid |>
  rowwise() |>
  mutate(power = {
    ps <- replicate(n_sim, simulate_one(n_speakers, n_items))
    mean(ps < .05, na.rm = TRUE)
  }) |>
  ungroup()

print(results)
write_csv(results, here::here("data", "processed", "ch02_power_results.csv"))
cat("完了。data/processed/ch02_power_results.csv に保存した。\n")


# 第3章 研究倫理とオープンサイエンス

## `ch03_p_hacking_simulation.R`

対象章: 第3章「研究倫理とオープンサイエンス」

「HARKingとp値ハッキング」節の作例。Simmons, Nelson, and Simonsohn (2011)
が指摘した「開示されない柔軟性」が、真に効果が存在しない場合でも
偽陽性率を名目上の5%から大きく引き上げてしまうことを、架空のデータへの
実際のシミュレーションで確認する。実測データではない。

設計: 真に効果がない(2群の母平均が同じ)状況で、以下の2つの
「柔軟性」を研究者が使えるとする。
  (a) 任意打ち切り: n=20で有意でなければ、n=40まで追加してから再検定する
  (b) 従属変数の選択: 相関のある2つの従属変数を測定し、p値の小さい方を報告する
これらを組み合わせた場合の偽陽性率を、単純な1回きりの検定と比較する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)
n_sim <- 5000

simulate_naive <- function() {
  # 単純な1回きりの検定(n=20、DV1本のみ)。真の効果はゼロ。
  a <- rnorm(20, 0, 1); b <- rnorm(20, 0, 1)
  t.test(a, b)$p.value
}

simulate_flexible <- function() {
  # (a) 任意打ち切り + (b) 従属変数の選択、を両方使う場合
  a1 <- rnorm(20, 0, 1); b1 <- rnorm(20, 0, 1)
  # DV2はDV1と相関r=0.5を持つよう生成(同じ参加者の別指標という想定)
  a2 <- 0.5 * a1 + rnorm(20, 0, sqrt(1 - 0.5^2))
  b2 <- 0.5 * b1 + rnorm(20, 0, sqrt(1 - 0.5^2))
  p1 <- t.test(a1, b1)$p.value
  p2 <- t.test(a2, b2)$p.value
  p_best_n20 <- min(p1, p2)
  if (p_best_n20 < .05) return(p_best_n20)

  # 有意でなければ、さらに20名追加してn=40で再検定
  a1b <- c(a1, rnorm(20, 0, 1)); b1b <- c(b1, rnorm(20, 0, 1))
  a2b <- c(a2, 0.5 * a1b[21:40] + rnorm(20, 0, sqrt(1 - 0.5^2)))
  b2b <- c(b2, 0.5 * b1b[21:40] + rnorm(20, 0, sqrt(1 - 0.5^2)))
  p1b <- t.test(a1b, b1b)$p.value
  p2b <- t.test(a2b, b2b)$p.value
  min(p1b, p2b)
}

p_naive    <- replicate(n_sim, simulate_naive())
p_flexible <- replicate(n_sim, simulate_flexible())

fpr_naive    <- mean(p_naive < .05)
fpr_flexible <- mean(p_flexible < .05)

cat("単純な1回きりの検定での偽陽性率:", round(fpr_naive * 100, 1), "%\n")
cat("2種の柔軟性を組み合わせた場合の偽陽性率:", round(fpr_flexible * 100, 1), "%\n")
cat("名目上の水準(5%)に対する倍率:", round(fpr_flexible / .05, 2), "倍\n")


# 第4章 音響分析の基礎

## `ch04_source_filter.R`

対象章: 第4章「音響分析の基礎」

「音源フィルタ理論と声道の共鳴」節の模式図。音源スペクトル(倍音列)、
声道の伝達関数(フォルマントのピーク)、そして両者を掛け合わせた出力
スペクトルを3パネルで示す、音源フィルタ理論の教科書的な図解。
理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

f0 <- 120
freq <- seq(0, 4000, by = 5)

# 音源スペクトル: 基本周波数の整数倍にピークを持つ倍音列(振幅は緩やかに減衰)
harmonics <- seq(f0, 4000, by = f0)
source_spec <- tibble(freq = freq, amp = 0)
for (h in harmonics) {
  source_spec$amp <- source_spec$amp + dnorm(source_spec$freq, mean = h, sd = 4) * (1 / (h / f0)^0.7)
}
source_spec$amp <- source_spec$amp / max(source_spec$amp)

# 声道の伝達関数: フォルマント(F1, F2, F3)に対応する共鳴ピーク
formants <- tibble(F = c(700, 1200, 2600), bw = c(70, 90, 130))
filter_resp <- tibble(freq = freq, gain = 0)
for (i in seq_len(nrow(formants))) {
  filter_resp$gain <- filter_resp$gain +
    1 / (1 + ((filter_resp$freq - formants$F[i]) / formants$bw[i])^2)
}
filter_resp$gain <- filter_resp$gain / max(filter_resp$gain)

output_spec <- tibble(freq = freq, amp = source_spec$amp * (0.15 + 0.85 * filter_resp$gain))
output_spec$amp <- output_spec$amp / max(output_spec$amp)

mk_panel <- function(df, yvar, title, colour) {
  ggplot(df, aes(x = freq, y = .data[[yvar]])) +
    geom_area(fill = colour, alpha = 0.25) +
    geom_line(colour = colour, linewidth = 0.7) +
    labs(title = title, x = "周波数 (Hz)", y = NULL) +
    scale_y_continuous(breaks = NULL) +
    theme(plot.title = element_text(size = 10, face = "bold"))
}

p1 <- mk_panel(source_spec, "amp", "音源スペクトル(声帯振動、倍音列)", "#34518A")
p2 <- mk_panel(filter_resp, "gain", "声道の伝達関数(フォルマント)", "#A8681F")
p3 <- mk_panel(output_spec, "amp", "出力スペクトル(音源×フィルタ)", "#5C7A3D")

library(patchwork)
p <- p1 / p2 / p3
ggsave(here::here("assets", "figures", "ch04", "fig04-01-source-filter.png"),
       p, width = 6, height = 6.5, dpi = 320)
ggsave(here::here("assets", "figures", "ch04", "fig04-01-source-filter.pdf"),
       p, width = 6, height = 6.5, device = cairo_pdf)
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_sampling_aliasing.R`

対象章: 第4章「音響分析の基礎」

「標本化・量子化とナイキスト周波数」節の模式図。連続的な正弦波を、
ナイキスト周波数を下回る標本化周波数でサンプリングすると、実際より
低い周波数の波形として偽って記録される(エイリアシング)ことを示す。
理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_freq <- 900     # 真の信号の周波数(Hz)
fs_ok     <- 8000    # 十分な標本化周波数(ナイキスト周波数4000Hz > 900Hz)
fs_bad    <- 1000    # 不十分な標本化周波数(ナイキスト周波数500Hz < 900Hz)
duration  <- 0.01     # 表示区間(秒)

t_cont <- seq(0, duration, length.out = 2000)
cont_wave <- tibble(t = t_cont, amp = sin(2 * pi * true_freq * t_cont))

sample_and_alias <- function(fs, label) {
  t_samp <- seq(0, duration, by = 1 / fs)
  tibble(t = t_samp, amp = sin(2 * pi * true_freq * t_samp), fs_label = label)
}
samp_ok  <- sample_and_alias(fs_ok, paste0("fs = ", fs_ok, "Hz(ナイキスト周波数 > 信号の周波数)"))
samp_bad <- sample_and_alias(fs_bad, paste0("fs = ", fs_bad, "Hz(ナイキスト周波数 < 信号の周波数)"))

# エイリアシングで見かけ上生じる低周波成分(標本点だけを滑らかに結んだ場合に見える周波数)
alias_freq <- abs(true_freq - fs_bad * round(true_freq / fs_bad))
t_alias <- seq(0, duration, length.out = 2000)
alias_wave <- tibble(t = t_alias, amp = sin(2 * pi * alias_freq * t_alias + pi/2.2))

p1 <- ggplot() +
  geom_line(data = cont_wave, aes(t, amp), colour = "grey60", linewidth = 0.5) +
  geom_point(data = samp_ok, aes(t, amp), colour = "#34518A", size = 2) +
  geom_line(data = samp_ok, aes(t, amp), colour = "#34518A", linewidth = 0.4, linetype = 2) +
  labs(title = unique(samp_ok$fs_label), x = NULL, y = "振幅") +
  theme(plot.title = element_text(size = 9))

p2 <- ggplot() +
  geom_line(data = cont_wave, aes(t, amp), colour = "grey60", linewidth = 0.5) +
  geom_point(data = samp_bad, aes(t, amp), colour = "#A8681F", size = 2.5) +
  geom_line(data = alias_wave, aes(t, amp), colour = "#A8681F", linewidth = 0.7, linetype = 2) +
  labs(title = paste0(unique(samp_bad$fs_label), " → 見かけ上", round(alias_freq), "Hzとして記録される"),
       x = "時間 (s)", y = "振幅") +
  theme(plot.title = element_text(size = 9))

library(patchwork)
p <- p1 / p2
ggsave(here::here("assets", "figures", "ch04", "fig04-02-sampling-aliasing.png"),
       p, width = 6.5, height = 5, dpi = 320)
ggsave(here::here("assets", "figures", "ch04", "fig04-02-sampling-aliasing.pdf"),
       p, width = 6.5, height = 5, device = cairo_pdf)
cat("エイリアシングで見かけ上生じる周波数:", round(alias_freq), "Hz\n")
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_lpc_order.R`

対象章: 第4章「音響分析の基礎」

「LPCによるフォルマント推定の原理と注意点」節の作例。既知のフォルマント
(F1=750Hz, F2=1200Hz付近、母音/a/を想定)を持つ合成母音を作成し、
LPC次数を3段階(低い・適切・高い)に変えて、実際にLPC分析を行い、
次数がフォルマント候補の推定にどう影響するかを確認する。
phonToolsパッケージのvowelsynth()による合成音声であり、実録音ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(phonTools)

fs_hz <- 11000
true_f1 <- 750
true_f2 <- 1200

set.seed(20260906)   # vowelsynth()の雑音成分(noise2)を固定し、再実行で同じ結果を得る

snd <- vowelsynth(ffs = c(true_f1, true_f2, 2800, 3400), fbw = 0.08,
                   dur = 300, f0 = c(120, 110), fs = fs_hz, noise2 = 0.01)

# LPC係数から極(フォルマント候補)の周波数と帯域幅を求める
lpc_to_formants <- function(a, fs) {
  rts <- polyroot(a)
  rts <- rts[Im(rts) > 0]
  ang <- atan2(Im(rts), Re(rts))
  freqs <- ang * fs / (2 * pi)
  bw <- -fs / pi * log(abs(rts))
  keep <- bw < 400 & freqs > 50   # 帯域幅が狭く、直流成分でない極だけを候補とする
  tibble(freq = freqs[keep], bw = bw[keep]) |> arrange(freq)
}

orders <- c(6, 13, 26)
results <- map(orders, function(ord) {
  a <- lpc(snd, order = ord, fs = fs_hz, show = FALSE)
  lpc_to_formants(a, fs_hz)
})
names(results) <- paste0("次数", orders)

for (nm in names(results)) {
  cat(nm, ": ", paste(round(results[[nm]]$freq), collapse = ", "), " Hz\n", sep = "")
}

cat("\n真のフォルマント(合成に使用): F1 =", true_f1, "Hz, F2 =", true_f2, "Hz\n")

# --- スペクトル包絡の比較図 ------------------------------------------------
freq_axis <- seq(50, 5000, by = 10)
lpc_envelope <- function(a, fs, freqs) {
  z <- exp(-2i * pi * freqs / fs)
  h <- 1 / sapply(z, function(zz) sum(a * zz^(0:(length(a) - 1))))
  20 * log10(abs(h))
}

spec <- spectralslice(snd, fs = fs_hz, show = FALSE)
spec_df <- tibble(freq = spec[, 1], db = spec[, 2]) |> filter(freq <= 5000)

env_df <- map_dfr(orders, function(ord) {
  a <- lpc(snd, order = ord, fs = fs_hz, show = FALSE)
  env <- lpc_envelope(a, fs_hz, freq_axis)
  tibble(order = paste0("次数", ord), freq = freq_axis, db = env - max(env) + max(spec_df$db) - 5)
})

p <- ggplot() +
  geom_line(data = spec_df, aes(freq, db), colour = "grey70", linewidth = 0.3) +
  geom_line(data = env_df, aes(freq, db, colour = order, linetype = order), linewidth = 0.9) +
  geom_vline(xintercept = c(true_f1, true_f2), linetype = 2, colour = "grey40") +
  scale_colour_manual(values = c("次数6" = "#A8681F", "次数13" = "#34518A", "次数26" = "#5C7A3D")) +
  scale_linetype_manual(values = c("次数6" = "solid", "次数13" = "22", "次数26" = "42")) +
  labs(x = "周波数 (Hz)", y = "振幅 (dB)", colour = "LPC次数", linetype = "LPC次数",
       caption = "灰色は合成音声のスペクトル、破線は真のF1・F2。phonTools::vowelsynth()による合成音声。scripts/R/ch04_lpc_order.R より生成。")
save_fig(p, "fig04-04-lpc-order-comparison", chapter = "ch04", width = 7, height = 4)
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_vot_waveform.R`

対象章: 第4章「音響分析の基礎」

「持続時間・強度・スペクトルモーメント・VOTの測定」節の模式図。VOT
(Voice Onset Time)の定義を、閉鎖開放(バースト)から声帯振動開始
までの波形上で図示する。理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

fs <- 16000
t <- seq(0, 0.25, by = 1 / fs)

burst_time  <- 0.05   # 閉鎖解放(バースト)の時刻
voice_onset <- 0.09   # 声帯振動開始の時刻(正のVOT)

wave <- numeric(length(t))
# 閉鎖区間(バースト以前): ほぼ無音
wave[t < burst_time] <- rnorm(sum(t < burst_time), 0, 0.01)
# バースト: 短い広帯域ノイズのパルス
burst_idx <- which(t >= burst_time & t < burst_time + 0.003)
wave[burst_idx] <- rnorm(length(burst_idx), 0, 0.8) * exp(-((t[burst_idx] - burst_time) / 0.001)^2)
# 帯気区間(バースト後、声帯振動開始前): 減衰する乱流性ノイズ
asp_idx <- which(t >= burst_time + 0.003 & t < voice_onset)
wave[asp_idx] <- rnorm(length(asp_idx), 0, 0.15) * exp(-(t[asp_idx] - burst_time) / 0.02)
# 声帯振動開始後: 周期的な母音波形
voi_idx <- which(t >= voice_onset)
wave[voi_idx] <- 0.6 * sin(2 * pi * 120 * (t[voi_idx] - voice_onset)) *
  (1 - exp(-(t[voi_idx] - voice_onset) / 0.01)) +
  rnorm(length(voi_idx), 0, 0.02)

d <- tibble(t = t * 1000, amp = wave)   # ミリ秒に変換

p <- ggplot(d, aes(t, amp)) +
  geom_line(colour = "#34518A", linewidth = 0.3) +
  geom_vline(xintercept = burst_time * 1000, linetype = 2, colour = "#A8681F") +
  geom_vline(xintercept = voice_onset * 1000, linetype = 2, colour = "#5C7A3D") +
  annotate("text", x = burst_time * 1000, y = 0.95, label = "閉鎖解放(バースト)", size = 3, hjust = -0.05, colour = "#A8681F") +
  annotate("text", x = voice_onset * 1000, y = 0.75, label = "声帯振動開始", size = 3, hjust = -0.05, colour = "#5C7A3D") +
  annotate("segment", x = burst_time * 1000, xend = voice_onset * 1000, y = -0.9, yend = -0.9,
           arrow = arrow(ends = "both", length = unit(0.15, "cm")), colour = "grey30") +
  annotate("text", x = mean(c(burst_time, voice_onset)) * 1000, y = -1.05,
           label = paste0("VOT = ", round((voice_onset - burst_time) * 1000), "ms"), size = 3.2, colour = "grey20") +
  scale_y_continuous(limits = c(-1.15, 1.05)) +
  labs(x = "時間 (ms)", y = "振幅",
       caption = "模式波形(実測データではない)。scripts/R/ch04_vot_waveform.R より生成。")

save_fig(p, "fig04-05-vot-waveform", chapter = "ch04", width = 7, height = 3.6)
cat("VOT =", round((voice_onset - burst_time) * 1000), "ms\n")
cat("完了。図はassets/figures/ch04/に保存した。\n")


# 第5章 Praatによる音響分析

## `ch05_window_length.R`

対象章: 第5章「Praatによる音響分析」

「測定パラメータの設定と結果への影響」節の作例。フォルマント分析の
窓長(Praatの"window length"パラメータ)が、時間分解能と周波数分解能の
トレードオフをどう左右するかを、実際に計算して確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

window_lengths <- c(0.010, 0.025, 0.050)   # 秒

# 周波数分解能の目安: およそ 1 / 窓長(Hz)
freq_resolution <- 1 / window_lengths

# 時間分解能の目安: 窓の半分だけ前後にずれると値が変わりうる(ミリ秒換算)
time_resolution_ms <- window_lengths / 2 * 1000

d <- data.frame(
  window_length_s   = window_lengths,
  freq_resolution_hz = round(freq_resolution, 1),
  time_resolution_ms = time_resolution_ms
)
print(d)


## `ch05_pitch_range.R`

対象章: 第5章「Praatによる音響分析」

「測定パラメータの設定と結果への影響」節の作例。ピッチレンジ(floor/
ceiling)の設定が狭すぎる・広すぎる場合に、真のF0がその範囲を外れる
ことでオクターブエラーが生じる様子を、架空のF0軌跡で確認する。
実測データではない。自己相関法に基づくピッチ追跡アルゴリズムでは、
真のF0がceilingを超えると、その半分の周波数(1オクターブ下)が
誤って検出されやすいことが知られている。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_points <- 500
t <- seq(0, 2, length.out = n_points)

# 自然な発話のF0軌跡を模した架空の曲線: 全体的な下降(declination)に、
# 強調によるピーク(260Hz前後まで上昇)と、句末の低下(60Hz台まで下降)を重ねる。
true_f0 <- 170 - 30 * t +
  110 * exp(-((t - 0.6) / 0.15)^2) +     # 強調によるピーク
  -50 * exp(-((t - 1.8) / 0.1)^2) +      # 句末の低下
  rnorm(n_points, 0, 4)

settings <- tibble::tribble(
  ~setting,   ~floor, ~ceiling,
  "狭い",      100,     250,
  "標準",      75,      350,
  "広い",      50,      600
)

# 設定ごとに、真のF0が範囲外になる(オクターブエラーが生じうる)割合を計算し、
# 範囲外の点について、典型的な自己相関法の挙動(ceiling超過は半分の周波数、
# floor未満は追跡失敗)を模して「測定される」F0を生成する。
simulate_measured <- function(floor, ceiling) {
  measured <- true_f0
  above <- true_f0 > ceiling
  below <- true_f0 < floor
  measured[above] <- true_f0[above] / 2       # ceiling超過 → 半分の周波数として誤検出
  measured[below] <- NA                        # floor未満 → 追跡失敗(undefined)
  list(measured = measured, pct_error = mean(above | below) * 100)
}

results <- settings |>
  rowwise() |>
  mutate(pct_error = simulate_measured(floor, ceiling)$pct_error) |>
  ungroup()

print(results)

d_plot <- purrr::pmap_dfr(settings, function(setting, floor, ceiling) {
  m <- simulate_measured(floor, ceiling)$measured
  tibble(setting = setting, t = t, f0 = m)
}) |>
  mutate(setting = factor(setting, levels = c("狭い", "標準", "広い")))

d_true <- tibble(t = t, f0 = true_f0)

p <- ggplot(d_plot, aes(t, f0)) +
  geom_line(data = d_true, aes(t, f0), colour = "grey70", linewidth = 0.4) +
  geom_point(size = 0.6, colour = "#A8681F") +
  facet_wrap(~setting, nrow = 1) +
  labs(x = "時間 (s)", y = "F0 (Hz)",
       caption = "灰色は真のF0軌跡、点は各ピッチレンジ設定での測定値(シミュレーション)。scripts/R/ch05_pitch_range.R より生成。")
save_fig(p, "fig05-03-pitch-range-comparison", chapter = "ch05", width = 9, height = 3)

cat("完了。図はassets/figures/ch05/に保存した。\n")


## `ch05_boundary_agreement.R`

対象章: 第5章「Praatによる音響分析」

「アノテーター間の一致率」節の作例。境界時刻のような連続的な測定について、
2人のアノテーターの一致を平均絶対誤差(MAE)で報告する場合、ランダムな
ばらつきだけでなく系統的なずれ(バイアス)がある場合にMAEがどう振る舞うか
を、実際にシミュレーションで確認する。架空のデータであり、実測データ
ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_boundaries <- 100
true_boundary <- rnorm(n_boundaries, 500, 80)   # ミリ秒、真の境界時刻(架空)

# ケース1: ランダムなばらつきのみ(系統的なずれなし)
annot_a_random <- true_boundary + rnorm(n_boundaries, 0, 8)
annot_b_random <- true_boundary + rnorm(n_boundaries, 0, 8)
diff_random <- annot_a_random - annot_b_random

# ケース2: アノテーターBが常に一定時間早く境界を引く(系統的バイアス)
annot_a_biased <- true_boundary + rnorm(n_boundaries, 0, 8)
annot_b_biased <- true_boundary - 15 + rnorm(n_boundaries, 0, 8)   # 常に15ms早い
diff_biased <- annot_a_biased - annot_b_biased

cat("--- ランダムなばらつきのみ ---\n")
cat("MAE:", round(mean(abs(diff_random)), 1), "ms  平均差(符号つき):",
    round(mean(diff_random), 1), "ms\n")

cat("\n--- 系統的バイアスあり(Bが常に15ms早い) ---\n")
cat("MAE:", round(mean(abs(diff_biased)), 1), "ms  平均差(符号つき):",
    round(mean(diff_biased), 1), "ms\n")


# 第6章 分析の自動化

## `ch06_pipeline.R`

対象章: 第6章「分析の自動化」

「Praat Scriptの基礎」節のパイプライン全体図。生の音声ファイルから
測定値テーブルに至るまでの処理の流れを、本章が扱う技術要素
(Praat Script、Parselmouth、強制アラインメント、品質管理)ごとに
図示する概念図。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

nodes <- tibble(
  x     = 1,
  y     = 6:1,
  label = c(
    "音声ファイル群(.wav)\n+ 書き起こしテキスト",
    "強制アラインメント(MFA)\n→ 音素・単語ごとのTextGrid",
    "Praat Script / Parselmouth\nバッチ測定(F0・フォルマント・持続時間等)",
    "自動測定の品質管理\n(異常値検出、信頼度の低い区間の除外)",
    "処理の来歴の記録\n(バージョン・パラメータ・実行日時)",
    "測定値テーブル(CSV)\n→ 第9章以降の統計解析へ"
  )
)

p <- ggplot(nodes) +
  geom_rect(aes(xmin = 0, xmax = 10, ymin = y - 0.44, ymax = y + 0.44),
            fill = "#34518A", alpha = 0.12, colour = "#34518A") +
  geom_text(aes(x = 5, y = y, label = label), size = 4.6, colour = "#1B2333", lineheight = 1.0) +
  geom_segment(data = nodes |> filter(y > 1),
               aes(x = 5, xend = 5, y = y - 0.47, yend = y - 0.73),
               arrow = arrow(length = unit(0.2, "cm"), type = "closed"), colour = "#A8681F", linewidth = 0.6) +
  scale_x_continuous(limits = c(0, 10)) +
  scale_y_continuous(limits = c(0.4, 6.6)) +
  theme_void() +
  labs(caption = "概念図。scripts/R/ch06_pipeline.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig06-01-pipeline-diagram", chapter = "ch06", width = 6, height = 8)
cat("完了。図はassets/figures/ch06/に保存した。\n")


## `ch06_mfa_bias_propagation.R`

対象章: 第6章「分析の自動化」

「アラインメント精度の限界と検証」節の作例。MFAの境界誤差が音素クラスに
よって系統的に異なる場合(閉鎖音は誤差が小さくランダム、摩擦音は
誤差が大きく系統的に短く出る)、その誤差が下流の統計的結論(2つの
音素クラス間の持続時間の差)にどう伝播するかを、実際にシミュレーション
で確認する。真の持続時間はどちらのクラスも同一(100ms)に設定してあり、
観測される差はすべてMFAの境界誤差に由来する。架空のデータであり、
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 200
true_duration <- 100   # ミリ秒、両クラスで同一の真の持続時間

# 閉鎖音: MFAの境界誤差はランダムでほぼ0を中心とする
stop_measured <- true_duration + rnorm(n_tokens, 0, 5)

# 摩擦音: 遷移がなだらかなため、MFAは境界を早めに(短く)検出しがち
fricative_measured <- true_duration - 20 + rnorm(n_tokens, 0, 5)

cat("真の持続時間: 両クラスとも", true_duration, "ms(同一)\n\n")
cat("閉鎖音(MFA境界誤差はランダム): 測定平均 =",
    round(mean(stop_measured), 1), "ms\n")
cat("摩擦音(MFA境界誤差が系統的に-20ms): 測定平均 =",
    round(mean(fricative_measured), 1), "ms\n")

test <- t.test(fricative_measured, stop_measured)
cat("\n2群の測定値の差の検定: 差 =", round(diff(test$estimate) * -1, 1),
    "ms, p =", format.pval(test$p.value, digits = 3), "\n")
cat("(真の差は0msだが、MFAの境界誤差だけで統計的に有意な差が生じている)\n")


## `ch06_dict_coverage.R`

対象章: 第6章「分析の自動化」

「強制アラインメント(MFA)の原理と手順」節の作例。発音辞書が話し言葉の
縮約形をどれだけ収録しているかによって、実際に整合できるトークンの
割合がどう変わるかを、架空の語彙頻度分布で確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

set.seed(20260106)
n_tokens <- 2000

# 上位20語(高頻度語)は縮約形が生じやすく、残りの語彙は縮約が稀とする
word_rank <- sample(1:200, n_tokens, replace = TRUE,
                     prob = c(rep(5, 20), rep(1, 180)))
is_high_freq <- word_rank <= 20

# 縮約が生じる確率: 高頻度語で40%、それ以外で5%
contracted <- ifelse(is_high_freq, rbinom(n_tokens, 1, 0.40),
                                    rbinom(n_tokens, 1, 0.05))

dict_covers_contraction <- FALSE   # 標準的な発音辞書は縮約形を収録していないと仮定
aligned_ok <- !(contracted == 1 & !dict_covers_contraction)

cat("総トークン数:", n_tokens, "\n")
cat("縮約形が生じたトークン数:", sum(contracted), "(",
    round(mean(contracted) * 100, 1), "%)\n")
cat("縮約形のうち高頻度語(上位20語)由来の割合:",
    round(mean(word_rank[contracted == 1] <= 20) * 100, 1), "%\n")
cat("辞書に縮約形がないために整合できなかったトークン数:",
    sum(!aligned_ok), "(全体の", round(mean(!aligned_ok) * 100, 1), "%)\n")


## `ch06_outlier_detection.R`

対象章: 第6章「分析の自動化」

「自動測定の品質管理」節の作例。平均・標準偏差に基づく外れ値検出と、
中央値・MAD(中央値絶対偏差)に基づく外れ値検出が、外れ値そのものが
多く紛れ込むデータに対してどう異なる挙動を示すかを、架空のデータで
確認する。実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_normal  <- 260
n_extreme <- 60   # 測定失敗由来の極端な値(例: ピッチのオクターブエラー)。混入率を高めに設定

normal_vals  <- rnorm(n_normal, mean = 150, sd = 15)     # 正常なF0測定値(Hz)
extreme_vals <- rnorm(n_extreme, mean = 290, sd = 15)    # オクターブエラー由来の異常値(約2倍)

d <- tibble(f0 = c(normal_vals, extreme_vals),
            true_status = c(rep("正常", n_normal), rep("測定失敗", n_extreme)))

# --- 方法1: 平均±3SD -----------------------------------------------------
mean_sd_flag <- abs(d$f0 - mean(d$f0)) > 3 * sd(d$f0)

# --- 方法2: 中央値±3MAD(MADは正規分布に合わせて1.4826倍で標準化) --------
med <- median(d$f0)
mad_val <- mad(d$f0)   # Rのmad()はデフォルトで1.4826倍済み
mad_flag <- abs(d$f0 - med) > 3 * mad_val

d <- d |> mutate(mean_sd_flag = mean_sd_flag, mad_flag = mad_flag)

# 検出率(真の異常値のうち、実際に検出された割合)と誤検出率を集計
summarize_method <- function(flag) {
  tibble(
    検出された異常値数 = sum(flag & d$true_status == "測定失敗"),
    異常値の検出率     = sum(flag & d$true_status == "測定失敗") / n_extreme,
    正常値の誤検出数   = sum(flag & d$true_status == "正常"),
    誤検出率           = sum(flag & d$true_status == "正常") / n_normal
  )
}

cat("=== 平均±3SD法 ===\n")
print(summarize_method(d$mean_sd_flag))
cat("平均:", round(mean(d$f0), 1), " SD:", round(sd(d$f0), 1), "\n\n")

cat("=== 中央値±3MAD法 ===\n")
print(summarize_method(d$mad_flag))
cat("中央値:", round(med, 1), " MAD:", round(mad_val, 1), "\n")


## `ch06_checkpoint_resume.R`

対象章: 第6章「分析の自動化」

「処理の来歴とチェックポイント」節の作例。すでに処理済みのファイルを
記録しておき、パイプラインが中断してから再実行する際に、処理済みの
ファイルをスキップして未処理のファイルだけを処理する仕組みを、
架空のファイル一覧に対して実際に動かして確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

all_files <- sprintf("spk%02d_item%02d.wav", rep(1:10, each = 5), rep(1:5, 10))
cat("処理対象の総ファイル数:", length(all_files), "\n")

# 1回目の実行: 途中(37件目)で中断したと仮定する
processed_log <- all_files[1:37]
cat("1回目の実行で処理済み(中断まで):", length(processed_log), "件\n")

# 2回目の実行: ログに記録されたファイルをスキップし、残りだけを処理する
remaining <- setdiff(all_files, processed_log)
cat("2回目の実行でスキップされるファイル数:", length(all_files) - length(remaining), "\n")
cat("2回目の実行で新たに処理されるファイル数:", length(remaining), "\n")
cat("先頭3件:", paste(head(remaining, 3), collapse = ", "), "\n")


# 第7章 産出実験

## `ch07_recording_setup.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質の管理」節の模式図。防音環境における話者・
ヘッドセットマイク・録音機器の配置を、上から見た模式図として示す。
概念図であり、実際の部屋の実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

room <- tibble(xmin = 0, xmax = 10, ymin = 0, ymax = 6)

# 壁面の吸音材を示す短いハッチ線(内壁沿いに並べる)
wall_pad <- function(side, n = 14) {
  if (side %in% c("top", "bottom")) {
    x0 <- seq(0.3, 9.7, length.out = n)
    y0 <- if (side == "top") 5.85 else 0.15
    tibble(x = x0, xend = x0, y = y0 - 0.18, yend = y0 + 0.18)
  } else {
    y0 <- seq(0.3, 5.7, length.out = round(n * 0.6))
    x0 <- if (side == "left") 0.15 else 9.85
    tibble(x = x0 - 0.18, xend = x0 + 0.18, y = y0, yend = y0)
  }
}
pads <- bind_rows(wall_pad("top"), wall_pad("bottom"), wall_pad("left"), wall_pad("right"))

speaker_x <- 3.6; speaker_y <- 3
mic_x <- 3.6; mic_y <- 3.55
rec_x <- 8.0; rec_y <- 3

p <- ggplot() +
  # 部屋の外枠と吸音材ハッチ(防音・準防音環境であることを視覚的に示す)
  geom_rect(data = room, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax),
            fill = "#FBFAF7", colour = "#4A4A4A", linewidth = 1.1) +
  geom_segment(data = pads, aes(x = x, xend = xend, y = y, yend = yend),
               colour = "#B7AE9C", linewidth = 0.6) +
  annotate("label", x = 5, y = 5.6, label = "防音・準防音環境(壁面ハッチ=吸音材)",
           size = 3.4, colour = "grey25", fill = "white",
           linewidth = 0,
           fontface = "italic") +

  # 話者(頭部を丸、体を縦線で簡略化)
  geom_point(aes(x = speaker_x, y = speaker_y), size = 15, colour = "#34518A") +
  geom_point(aes(x = speaker_x, y = speaker_y), size = 15, shape = 1, colour = "white", stroke = 1.2) +
  annotate("text", x = speaker_x, y = speaker_y - 0.75, label = "話者",
           size = 4.6, fontface = "bold", colour = "#1B2333") +

  # ヘッドセットマイク(話者の口元、三角アイコン)。ラベルは真上に十分離して置き、
  # 引き出し線は直線1本だけにして、話者〜マイクの距離線や他の文字と交差させない。
  geom_point(aes(x = mic_x, y = mic_y), shape = 17, size = 6.5, colour = "#A8681F") +
  geom_segment(aes(x = mic_x + 0.55, xend = mic_x + 0.55, y = mic_y, yend = mic_y + 1.3),
               colour = "grey55", linewidth = 0.35) +
  annotate("text", x = mic_x + 0.75, y = mic_y + 1.3,
           label = "ヘッドセットマイク\n(口からの距離を一定に保つ)",
           size = 3.7, colour = "#1B2333", hjust = 0, vjust = 0.5, lineheight = 0.95) +

  # 話者〜マイク間の距離(本文の基準値5cmに対応する概念表示)。ラベルは距離線の
  # 真下に置き、話者アイコンや他のラベルと重ならないようにする。
  geom_segment(aes(x = speaker_x, xend = mic_x, y = speaker_y + 0.05, yend = mic_y - 0.05),
               linetype = 2, colour = "#7A1F1F", linewidth = 0.6) +
  annotate("label", x = speaker_x - 2.1, y = mic_y,
           label = "口からの距離\n(基準: 約5cm)", size = 3.5, colour = "#7A1F1F",
           fill = "white", linewidth = 0, fontface = "bold", lineheight = 0.9) +

  # 録音機器(角丸四角)
  geom_tile(aes(x = rec_x, y = rec_y), width = 1.3, height = 0.9,
            fill = "#5C7A3D", colour = "#3E5228", linewidth = 0.8) +
  annotate("text", x = rec_x, y = rec_y - 0.75, label = "録音機器\n(PC / レコーダー)",
           size = 4.0, fontface = "bold", colour = "#1B2333", lineheight = 0.9) +

  # マイク→録音機器のケーブル/信号経路
  geom_segment(aes(x = mic_x + 0.35, xend = rec_x - 0.75, y = mic_y, yend = rec_y),
               arrow = arrow(length = unit(0.22, "cm"), type = "closed"),
               colour = "grey40", linewidth = 0.6) +
  annotate("text", x = (mic_x + rec_x) / 2 + 0.2, y = (mic_y + rec_y) / 2 - 0.35,
           label = "信号", size = 3.3, colour = "grey40") +

  coord_fixed(xlim = c(-0.3, 10.3), ylim = c(-0.5, 6.3)) +
  theme_void() +
  labs(caption = "概念図(上から見た模式図)。マイク〜口の距離が広がるほどSNRが低下することを本文で数値例により示す。\nscripts/R/ch07_recording_setup.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.85), colour = "grey35", hjust = 0.5))

save_fig(p, "fig07-01-recording-setup", chapter = "ch07", width = 7.2, height = 4.6)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_mic_distance_snr.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質の管理」節の作例。マイクロフォンと話者の口との
距離が変動すると、音圧レベル(逆二乗則に基づく音圧の距離減衰)が
変化し、信号対雑音比(SNR)がどれだけ変わるかを、実際に計算して
確認する。理論的な計算例であり、実測データではない。
音圧は音源からの距離にほぼ反比例する(自由音場を仮定した理論値)

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

ref_distance <- 0.05     # 基準距離(ヘッドセットマイクの典型的な距離、5cm)
ref_spl_db   <- 70       # 基準距離での音圧レベル(dB SPL、仮定値)
noise_floor_db <- 35     # 録音環境の暗騒音レベル(dB SPL、仮定値)

distances <- c(0.05, 0.10, 0.30, 0.50)   # メートル: ヘッドセット, 近接, デスクマイク, 離れた設置

spl_at_distance <- ref_spl_db - 20 * log10(distances / ref_distance)
snr_at_distance <- spl_at_distance - noise_floor_db

d <- data.frame(distance_m = distances,
                spl_db = round(spl_at_distance, 1),
                snr_db = round(snr_at_distance, 1))
print(d)


## `ch07_clipping_detection.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質の管理」節の作例。クリッピング(振幅が記録可能な
上限を超えて飽和する現象)を、波形のサンプル値から自動的に検出する
簡便な方法を、架空の波形データで確認する。実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

fs <- 16000
dur <- 1
t <- seq(0, dur, length.out = fs * dur)

make_waveform <- function(gain) {
  raw <- gain * (sin(2 * pi * 150 * t) + 0.3 * sin(2 * pi * 450 * t) + rnorm(length(t), 0, 0.05))
  pmin(pmax(raw, -1), 1)   # -1から1の範囲で飽和(クリッピング)させる、16bit相当の正規化波形
}

detect_clipping <- function(wave, threshold = 0.999, min_run = 3) {
  is_ceiling <- abs(wave) >= threshold
  # 連続run長を計算し、min_run以上連続して天井に張り付いている箇所を検出する
  runs <- rle(is_ceiling)
  clipped_samples <- sum(runs$lengths[runs$values & runs$lengths >= min_run])
  clipped_samples / length(wave)
}

gains <- c(0.5, 0.9, 1.3, 2.0)
results <- map_dfr(gains, function(g) {
  wave <- make_waveform(g)
  tibble(gain = g, pct_clipped = detect_clipping(wave) * 100)
})

print(results)

d_plot <- map_dfr(gains, function(g) tibble(gain = g, t = t[1:800], amp = make_waveform(g)[1:800]))
p <- ggplot(d_plot, aes(t, amp)) +
  geom_line(colour = "#34518A", linewidth = 0.3) +
  geom_hline(yintercept = c(-1, 1), linetype = 2, colour = "#A8681F") +
  facet_wrap(~gain, labeller = label_both, nrow = 1) +
  labs(x = "時間 (s)", y = "振幅(正規化)",
       caption = "シミュレーション波形。破線は記録可能な振幅の上限・下限。scripts/R/ch07_clipping_detection.R より生成。")
save_fig(p, "fig07-02-clipping-detection", chapter = "ch07", width = 8, height = 2.6)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_lombard_diagnosis.R`

対象章: 第7章「産出実験」

「産出実験に固有のバイアス」節の作例。実験の丁寧発話(Lombard効果に
類する順応)が、試行順序に伴ってどう減衰していくかを、母音空間の
広さ(F1・F2から計算した凸包面積)を指標に、架空のデータで確認する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_trials  <- 80
# 丁寧発話の度合いは、実験開始直後にもっとも強く、指数関数的に減衰して
# 話者本来の(くだけた)発話パターンに近づいていくと想定する。
politeness_decay <- 25 * exp(-(1:n_trials) / 20)
vowel_space_area <- 180 + politeness_decay + rnorm(n_trials, 0, 8)

d <- tibble(trial = 1:n_trials, area = vowel_space_area)

m <- lm(area ~ trial, data = d)
slope <- coef(m)["trial"]
p_val <- summary(m)$coefficients["trial", "Pr(>|t|)"]

early_mean <- mean(d$area[d$trial <= 10])
late_mean  <- mean(d$area[d$trial > (n_trials - 10)])

cat("試行順の係数(1試行あたりの母音空間面積の変化):", round(slope, 3), "\n")
cat("p値:", format.pval(p_val, digits = 3), "\n")
cat("最初の10試行の平均面積:", round(early_mean, 1), "\n")
cat("最後の10試行の平均面積:", round(late_mean, 1), "\n")
cat("減少率:", round((early_mean - late_mean) / early_mean * 100, 1), "%\n")

p <- ggplot(d, aes(trial, area)) +
  geom_point(alpha = 0.5, size = 1.5, colour = "#34518A") +
  geom_smooth(method = "lm", se = TRUE, colour = "#A8681F", linewidth = 0.8, formula = y ~ x) +
  labs(x = "試行番号", y = "母音空間の面積(模式値)",
       caption = "シミュレーションデータ。scripts/R/ch07_lombard_diagnosis.R より生成。")
save_fig(p, "fig07-03-lombard-decay", chapter = "ch07", width = 6, height = 3.8)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_practice_trial_exclusion.R`

対象章: 第7章「産出実験」

「産出実験に固有のバイアス」節の作例。丁寧発話の減衰(ch07_lombard_
diagnosis.Rと同じ生成モデル)が残っている区間を含めたまま平均を
取ると、外れ値を除いた後も系統的なバイアスが残ることを、最初の
練習試行を除外した場合と除外しない場合とで比較して確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_trials <- 80
politeness_decay <- 25 * exp(-(1:n_trials) / 20)
vowel_space_area <- 180 + politeness_decay + rnorm(n_trials, 0, 8)
d <- tibble(trial = 1:n_trials, area = vowel_space_area)

true_baseline <- 180   # 丁寧発話の影響がなくなった後の真の値

n_practice <- 15
mean_all       <- mean(d$area)
mean_excluded  <- mean(d$area[d$trial > n_practice])

cat("真のベースライン(丁寧発話の影響なし):", true_baseline, "\n")
cat("全80試行の平均(練習試行を除外しない場合):", round(mean_all, 1),
    "  真値からの乖離:", round(mean_all - true_baseline, 1), "\n")
cat("最初の", n_practice, "試行を練習試行として除外した場合の平均:",
    round(mean_excluded, 1), "  真値からの乖離:",
    round(mean_excluded - true_baseline, 1), "\n")


# 第8章 知覚実験——刺激作成とPsychoPy

## `ch08_trial_structure.R`

対象章: 第8章「知覚実験——刺激作成とPsychoPy」

「PsychoPyによる実装」節の図解。同定課題の1試行が、時間軸上でどのような
区間(注視点→刺激提示→反応取得→試行間間隔)に分かれるかを示す模式図。
実測データではなく、試行構造そのものを説明するための概念図である。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

segments <- tibble(
  phase = c("注視点", "刺激提示", "反応取得(反応時間の起点)", "試行間間隔(ITI)"),
  start = c(0, 500, 700, 700 + 2000),
  end   = c(500, 700, 700 + 2000, 700 + 2000 + 500),
  y     = 1
) |>
  mutate(phase = factor(phase, levels = phase))

label_df <- segments |>
  mutate(mid = (start + end) / 2,
         # 白黒印刷でも読めるよう、濃い棒には白文字、薄い棒には黒文字にする
         txt = c("black", "white", "white", "black"))

p <- ggplot(segments) +
  geom_rect(aes(xmin = start, xmax = end, ymin = 0.3, ymax = 0.7, fill = phase), colour = "grey30") +
  geom_text(data = label_df |> filter(phase != "刺激提示"), aes(x = mid, y = 0.5, label = phase, colour = txt), size = 3.4, fontface = "bold") +
  geom_text(data = label_df |> filter(phase == "刺激提示"), aes(x = mid, y = 0.77, label = phase), size = 3.4, fontface = "bold", colour = "black") +
  scale_colour_identity() +
  geom_segment(data = label_df, aes(x = start, xend = start, y = 0.25, yend = 0.3), colour = "grey40") +
  geom_text(data = label_df, aes(x = start, y = 0.15, label = paste0(start, "ms")), size = 3, colour = "grey20") +
  annotate("text", x = tail(segments$end, 1), y = 0.04,
           label = paste0(tail(segments$end, 1), "ms(次の試行開始)"), size = 3, colour = "grey20", hjust = 1) +
  scale_fill_manual(values = c("注視点" = "#D2D2D2", "刺激提示" = "#2B2B2B",
                                "反応取得(反応時間の起点)" = "#6B6B6B", "試行間間隔(ITI)" = "#EBEBEB"),
                     guide = "none") +
  scale_x_continuous(limits = c(-50, tail(segments$end, 1) + 50)) +
  scale_y_continuous(limits = c(0, 0.85)) +
  theme_void() +
  labs(caption = "模式図(時間の目盛りは説明用の一例)。scripts/R/ch08_trial_structure.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig08-02-trial-structure", chapter = "ch08", width = 7, height = 2.2)
cat("完了。図はassets/figures/ch08/に保存した。\n")


## `ch08_headphone_check.R`

対象章: 第8章「知覚実験——刺激作成とPsychoPy」

「オンライン実験と計測上の制約」節の作例。位相を利用したヘッドホン
チェックで、何試行中何試行以上正解を基準とすれば、スピーカー使用者
(正答はほぼ偶然、3択なので正答率1/3と仮定)を十分な確率で不合格に
できるかを、二項分布に基づいて実際に計算する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_trials <- 6
p_headphone <- 0.90   # ヘッドホン使用者が単試行で正答する確率(仮定値)
p_speaker   <- 1 / 3  # スピーカー使用者が単試行で正答する確率(3択の偶然水準)

for (pass_threshold in c(4, 5, 6)) {
  pass_rate_headphone <- 1 - pbinom(pass_threshold - 1, n_trials, p_headphone)
  pass_rate_speaker   <- 1 - pbinom(pass_threshold - 1, n_trials, p_speaker)
  cat(n_trials, "試行中", pass_threshold, "試行以上正解を合格基準とする場合:\n")
  cat("  ヘッドホン使用者の合格率:", round(pass_rate_headphone * 100, 1), "%\n")
  cat("  スピーカー使用者が誤って合格する確率:", round(pass_rate_speaker * 100, 1), "%\n\n")
}


## `ch08_dprime_worked_example.R`

対象章: 第8章「知覚実験——刺激作成とPsychoPy」

「信号検出理論の基礎」節の作例。ヒット率とフォールスアラーム率から
d'(感度)とc(反応バイアス)を実際に計算する、具体的な数値例。
架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

hit_rate <- 0.82
fa_rate  <- 0.28

z_hit <- qnorm(hit_rate)
z_fa  <- qnorm(fa_rate)

d_prime <- z_hit - z_fa
criterion_c <- -0.5 * (z_hit + z_fa)

cat("ヒット率:", hit_rate, " フォールスアラーム率:", fa_rate, "\n")
cat("z(ヒット率) =", round(z_hit, 3), "  z(フォールスアラーム率) =", round(z_fa, 3), "\n")
cat("d' =", round(d_prime, 3), "\n")
cat("c  =", round(criterion_c, 3), "\n")

# 同じd'でも反応バイアスが異なる2人目の聞き手(cが大きい、慎重な反応)
hit_rate_2 <- 0.65
fa_rate_2  <- pnorm(qnorm(hit_rate_2) - d_prime)
cat("\n同じ感度(d' =", round(d_prime, 3), ")で、より慎重な反応基準を持つ聞き手の例:\n")
cat("ヒット率:", round(hit_rate_2, 3), " フォールスアラーム率:", round(fa_rate_2, 3), "\n")


## `ch08_sdt_roc.R`

対象章: 第8章「知覚実験——刺激作成とPsychoPy」

「信号検出理論の基礎」節の図解。ROC空間(横軸フォールスアラーム率、
縦軸ヒット率)上に、感度d'を一定に保ったまま反応バイアス(基準c)だけを
動かしたときの理論的なROC曲線と、ある1点の観測値を重ね、d'とcが
それぞれ何を表しているかを図示する。等分散正規分布を仮定する標準的な
SDTの定式化に基づく理論図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d_prime_example <- 1.5

# 基準cを動かしたときのROC曲線(理論値、Macmillan & Creelmanの標準的な定式化):
# z(H) = d'/2 - c, z(F) = -d'/2 - c
c_seq <- seq(-3, 3, length.out = 300)
roc_curve <- tibble(
  c = c_seq,
  F = pnorm(-d_prime_example / 2 - c_seq),
  H = pnorm(d_prime_example / 2 - c_seq)
)

# 観測値の例(基準がやや保守的に偏っている場合)
c_obs <- 0.3
obs <- tibble(F = pnorm(-d_prime_example / 2 - c_obs), H = pnorm(d_prime_example / 2 - c_obs))
d_prime_obs <- qnorm(obs$H) - qnorm(obs$F)
c_obs_est   <- -0.5 * (qnorm(obs$H) + qnorm(obs$F))

p <- ggplot(roc_curve, aes(F, H)) +
  geom_abline(slope = 1, intercept = 0, linetype = 3, colour = "grey50") +
  geom_line(colour = "#34518A", linewidth = 1) +
  geom_point(data = obs, aes(F, H), size = 3.5, colour = "#A8681F") +
  geom_segment(data = obs, aes(x = F, xend = F, y = 0, yend = H), linetype = 2, colour = "#A8681F", linewidth = 0.4) +
  geom_segment(data = obs, aes(x = 0, xend = F, y = H, yend = H), linetype = 2, colour = "#A8681F", linewidth = 0.4) +
  annotate("text", x = 0.62, y = 0.35, label = "d'一定のROC曲線\n(反応バイアスcだけが変化)",
           size = 3.2, colour = "#34518A", hjust = 0) +
  annotate("text", x = obs$F + 0.03, y = obs$H - 0.08,
           label = paste0("観測値の例\nF=", round(obs$F, 2), ", H=", round(obs$H, 2),
                           "\nd'=", round(d_prime_obs, 2), ", c=", round(c_obs_est, 2)),
           size = 3, colour = "#A8681F", hjust = 0) +
  scale_x_continuous(limits = c(0, 1), labels = scales::percent, name = "フォールスアラーム率(F)") +
  scale_y_continuous(limits = c(0, 1), labels = scales::percent, name = "ヒット率(H)") +
  labs(caption = "理論図(等分散正規分布モデル)。scripts/R/ch08_sdt_roc.R より生成。")

save_fig(p, "fig08-03-sdt-roc", chapter = "ch08", width = 5.5, height = 5)
cat("d'(観測点から逆算):", round(d_prime_obs, 3), " c(観測点から逆算):", round(c_obs_est, 3), "\n")
cat("完了。図はassets/figures/ch08/に保存した。\n")


# 第9章 R入門

## `ch09_vowel_space.R`

対象章: 第9章「R入門」

「ggplot2の文法と音声研究の定番図」節の母音空間図。日本語5母音
(/a i u e o/)のF1・F2をシミュレートし、本文に示したコードと同じ
描画パターン(F1・F2逆軸+楕円)で母音空間図を実際に生成する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260905)

# 日本語5母音のおおよその調音的位置づけに基づく、説明用のF1・F2中心値。
# 実測の音響値ではなく、母音空間図の構造(狭母音は下方、前舌母音は右方)
# を示すための仮の値である。
vowel_centers <- tibble(
  vowel = c("a", "i", "u", "e", "o"),
  F1    = c(750, 300, 350, 500, 450),
  F2    = c(1300, 2300, 1200, 1900, 900)
)

n_tokens <- 30
formants_long <- vowel_centers |>
  tidyr::crossing(token = 1:n_tokens) |>
  mutate(
    F1 = F1 + rnorm(n(), 0, 40),
    F2 = F2 + rnorm(n(), 0, 90)
  ) |>
  select(vowel, token, F1, F2)

p <- ggplot(formants_long, aes(x = F2, y = F1, color = vowel, shape = vowel)) +
  geom_point(alpha = 0.4, size = 1) +
  stat_ellipse(level = 0.68) +
  scale_x_reverse() +
  scale_y_reverse() +
  labs(x = "F2 (Hz)", y = "F1 (Hz)", color = "母音", shape = "母音",
       caption = "シミュレーションデータ。scripts/R/ch09_vowel_space.R より生成。") +
  theme_labphon()

save_fig(p, "fig09-01-vowel-space", chapter = "ch09")
cat("完了。図はassets/figures/ch09/に保存した。\n")


## `ch09_join_pitfalls.R`

対象章: 第9章「Rによるデータ処理と可視化」

「よくあるエラーとその対処」節の作例。(1) NAの伝播が要約統計量に
どう波及するか、(2) 結合キーの表記ゆれ(大文字・小文字)が left_join()
でどれだけの行を静かに欠落させるかを、実際にコードを実行して確認する。
架空のデータであり、実測データではない。
--- (1) NAの伝播 -----------------------------------------------------------

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

durations <- tibble(
  speaker  = rep(c("s01", "s02", "s03"), each = 4),
  duration = c(0.12, 0.14, 0.11, 0.13,
               0.15, NA,   0.16, 0.14,
               0.10, 0.11, 0.12, 0.09)
)

summary_no_narm <- durations |>
  group_by(speaker) |>
  summarize(mean_duration = mean(duration), .groups = "drop")

summary_narm <- durations |>
  group_by(speaker) |>
  summarize(mean_duration = mean(duration, na.rm = TRUE), .groups = "drop")

cat("na.rm指定なし:\n"); print(summary_no_narm)
cat("\nna.rm = TRUE:\n"); print(summary_narm)

# --- (2) 結合キーの表記ゆれ --------------------------------------------------
speakers_meta <- tibble(
  speaker = c("sp01", "sp02", "sp03", "sp04", "sp05"),
  gender  = c("F", "M", "F", "M", "F")
)

# 測定データ側で1名だけ大文字表記になっている(入力ミスを模す)
measures <- tibble(
  speaker  = c("sp01", "sp02", "SP03", "sp04", "sp05"),
  f0_mean  = c(210, 118, 205, 122, 198)
)

joined_naive <- measures |> left_join(speakers_meta, by = "speaker")
n_unmatched_naive <- sum(is.na(joined_naive$gender))

joined_fixed <- measures |>
  mutate(speaker = str_to_lower(speaker)) |>
  left_join(speakers_meta, by = "speaker")
n_unmatched_fixed <- sum(is.na(joined_fixed$gender))

cat("\n表記ゆれを正規化する前の未結合行数:", n_unmatched_naive, "/", nrow(measures), "\n")
cat("str_to_lower()で正規化した後の未結合行数:", n_unmatched_fixed, "/", nrow(measures), "\n")


## `ch09_seed_reproducibility.R`

対象章: 第9章「Rによるデータ処理と可視化」

「再現性のための仕組み」節の作例。乱数の種(set.seed())の有無で、
同じブートストラップ処理を2回実行した結果が一致するかどうかを、
実際に確認する。架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

toy_data <- tibble(f0 = c(210, 195, 220, 205, 198, 215, 202, 208))

bootstrap_mean <- function() {
  resampled <- sample(toy_data$f0, replace = TRUE)
  mean(resampled)
}

cat("--- シード固定なし ---\n")
run1_nofix <- bootstrap_mean()
run2_nofix <- bootstrap_mean()
cat("1回目:", run1_nofix, " 2回目:", run2_nofix,
    " 一致:", identical(run1_nofix, run2_nofix), "\n")

cat("\n--- set.seed(20260101)で固定 ---\n")
set.seed(20260101); run1_fixed <- bootstrap_mean()
set.seed(20260101); run2_fixed <- bootstrap_mean()
cat("1回目:", run1_fixed, " 2回目:", run2_fixed,
    " 一致:", identical(run1_fixed, run2_fixed), "\n")


## `ch09_csj_id_parsing.R`

対象章: 第9章「Rによるデータ処理と可視化」

「音声データに固有の整形とID設計」節の作例。CSJの講演ID(接頭辞+通し番号)
から講演種別を復元する処理を、架空のID一覧に対して実際に実行する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr); library(stringr)

talk_ids <- tibble(
  talk_id = c("A01M0007", "S02F1234", "A03M0102", "S04F0056")
)

talk_ids |>
  mutate(
    prefix = str_sub(talk_id, 1, 1),
    talk_type = case_when(
      prefix == "A" ~ "学会講演",
      prefix == "S" ~ "模擬講演",
      TRUE          ~ "不明"
    )
  ) |>
  print()


## `ch09_missingness_bias.R`

対象章: 第9章「Rによるデータ処理と可視化」

「欠測の記録」節の作例。欠測が言語現象と結びついている場合
(無声化した母音でF0が測定できない、など)、欠測行を静かに除外した
解析(リストワイズ除去)が、条件間の平均差をどれだけ歪めるかを、
架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

set.seed(20260906)
n <- 300

# 条件Bでは無声化が起きやすく(30%)、無声化するとF0が測定できない(NA)。
# 無声化したトークンは(仮に測定できたとしても)F0がやや低い母音だったとする。
condition <- rep(c("A", "B"), each = n / 2)
devoiced_prob <- ifelse(condition == "A", 0.05, 0.30)
devoiced <- rbinom(n, 1, devoiced_prob)
true_f0 <- 150 + ifelse(condition == "B", 0, 0) + rnorm(n, 0, 10) - devoiced * 8

d <- tibble(condition, devoiced, f0 = ifelse(devoiced == 1, NA, true_f0))

naive_means <- d |> group_by(condition) |> summarize(mean_f0 = mean(f0, na.rm = TRUE))
true_means  <- tibble(condition, true_f0) |> group_by(condition) |>
  summarize(mean_true_f0 = mean(true_f0))

cat("リストワイズ除去後の条件別平均F0(欠測を無視):\n")
print(naive_means)
cat("\n全トークン(欠測なし)での真の平均F0:\n")
print(true_means)
cat("\n条件間の差 --- 観測(欠測除去後):",
    round(diff(naive_means$mean_f0), 2),
    " / 真の値(欠測なし):", round(diff(true_means$mean_true_f0), 2), "\n")


# 第10章 統計的推論・回帰分析・混合効果モデル

## `sim_measures.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

scripts/R/10_mixed_models.R が読み込む data/processed/measures.csv を
シミュレートする。**実測データではない。** 話者20名×項目30語×条件2水準
(例: 通常発話 vs. 明瞭発話)の交差デザインを想定し、条件の主効果・
対数頻度の主効果・条件と頻度の交互作用を、話者・項目のランダム切片・
ランダム傾きとともに設定する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_speakers <- 20
n_items    <- 30

speakers <- tibble(
  speaker    = sprintf("sp%02d", 1:n_speakers),
  spk_int_re = rnorm(n_speakers, 0, 18),
  spk_slope_re = rnorm(n_speakers, 0, 6)
)
items <- tibble(
  item        = sprintf("item%02d", 1:n_items),
  frequency   = round(exp(runif(n_items, 0, 6))),
  item_int_re = rnorm(n_items, 0, 14)
)

true_intercept   <- 230   # 基準条件・平均頻度での持続時間(ms)
true_cond_effect <- 22    # 条件の主効果(和対比、片側の効果量)
true_freq_effect <- -9    # 対数頻度1単位あたりの持続時間の変化
true_interaction <- 5     # 条件×対数頻度の交互作用

d <- speakers |>
  tidyr::crossing(items) |>
  tidyr::crossing(condition = c("通常", "明瞭")) |>
  mutate(
    logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)),
    cond_c    = if_else(condition == "明瞭", 0.5, -0.5),
    duration_mean = true_intercept +
      true_cond_effect * cond_c +
      (true_freq_effect + spk_slope_re) * logfreq_c +
      true_interaction * cond_c * logfreq_c +
      spk_int_re + item_int_re,
    duration = pmax(duration_mean + rnorm(n(), 0, 20), 40)
  ) |>
  select(speaker, item, condition, frequency, duration)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "measures.csv"))
cat("生成した行数:", nrow(d), " / 話者数:", n_speakers, " / 項目数:", n_items, "\n")


## `10_mixed_models.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

なぜ混合効果モデルを使うのか:
  音声データは、同じ話者が複数の項目を、同じ項目を複数の話者が産出する
  という交差構造を持つ。話者ごと・項目ごとの平均差を無視して回帰すると、
  独立性の仮定が壊れ、第一種の過誤が膨らむ。ランダム効果は
  「この話者はもともと速い」「この語はもともと長い」を明示的にモデルに
  入れることで、固定効果の推定を話者・項目の偶然から切り離す。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d <- read_csv(here::here("data", "processed", "measures.csv"),
              na = "NA", show_col_types = FALSE) |>
  mutate(
    speaker  = factor(speaker),
    item     = factor(item),
    # 対比コーディング: 交互作用を含むモデルでは処理コーディングの
    # 「主効果」は参照水準での単純効果になる。和対比にしておくと
    # 主効果を平均的な効果として解釈できる（第10章 §対比コーディング）。
    condition = factor(condition)
  )
contrasts(d$condition) <- contr.sum(nlevels(d$condition))

# 連続予測変数は中心化する。切片の解釈が可能になり、収束もしやすくなる。
d <- d |> mutate(logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)))

# --- 1. maximal model から始める ---------------------------------------
# Barr et al. (2013) の「デザインで正当化される最大構造」。
# 収束しない場合に何を削るかは、恣意的に決めずに手順を決めておく（下記）。
m_max <- lmer(
  duration ~ condition * logfreq_c +
    (1 + condition | speaker) +
    (1 + condition | item),
  data = d, REML = TRUE,
  control = lmerControl(optimizer = "bobyqa",
                        optCtrl = list(maxfun = 2e5))
)

# --- 2. 収束しない場合の縮退手順（あらかじめ決めておく） ----------------
# (a) 相関パラメータを外す  (1 + condition || speaker)
# (b) 項目側のランダム傾きを外す
# (c) 話者側のランダム傾きを外す
# どこまで落としたかは必ず論文に書く。「収束しなかったので lm にした」は不可。
m_red <- lmer(
  duration ~ condition * logfreq_c +
    (1 + condition || speaker) +
    (1 | item),
  data = d, REML = TRUE, control = lmerControl(optimizer = "bobyqa")
)

# --- 3. 診断 ------------------------------------------------------------
print(summary(m_max))
cat("\n特異適合か: ", isSingular(m_max), "\n")

# 残差診断。正規性そのものより、系統的なパターンの有無を見る。
plot_resid <- tibble(fit = fitted(m_max), res = resid(m_max)) |>
  ggplot(aes(fit, res)) +
  geom_point(alpha = 0.25, size = 0.8) +
  geom_hline(yintercept = 0, linetype = 2) +
  geom_smooth(method = "loess", se = FALSE, linewidth = 0.6, formula = y ~ x) +
  labs(x = "予測値 (ms)", y = "残差 (ms)",
       caption = "scripts/R/10_mixed_models.R より生成")
save_fig(plot_resid, "fig10-02-residuals", chapter = "ch10")

# --- 4. 報告 ------------------------------------------------------------
# p 値だけを報告しない。推定値・標準誤差・信頼区間を併記する（style-guide §6）。
ci <- confint(m_max, method = "Wald", parm = "beta_")
print(cbind(Estimate = fixef(m_max), ci))


## `ch10_worked_example.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」通しの作例

「線形モデル・対比コーディング・交互作用」節の処理コーディング対比図と、
「混合効果モデル」節の残差診断プロットは、いずれも
data/processed/measures.csv (scripts/R/sim_measures.R が生成する架空データ)
を実際に解析した結果である。ここでは処理コーディングと和対比コーディングの
係数を並べる部分だけを担当する(残差診断はscripts/R/10_mixed_models.Rが担当)。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

d <- read_csv(here::here("data", "processed", "measures.csv"), show_col_types = FALSE) |>
  mutate(condition = factor(condition, levels = c("通常", "明瞭")))

d_treat <- d; contrasts(d_treat$condition) <- contr.treatment(2)
d_sum   <- d; contrasts(d_sum$condition)   <- contr.sum(2)

m_treat <- lm(duration ~ condition, data = d_treat)
m_sum   <- lm(duration ~ condition, data = d_sum)

coef_table <- tibble(
  coding = c("処理コーディング", "処理コーディング", "和対比コーディング", "和対比コーディング"),
  term   = c("(Intercept)", "condition2", "(Intercept)", "condition1"),
  estimate = c(coef(m_treat), coef(m_sum))
)
print(coef_table)

cat("\n通常条件の平均:", round(mean(d$duration[d$condition == "通常"]), 1), "ms\n")
cat("明瞭条件の平均:", round(mean(d$duration[d$condition == "明瞭"]), 1), "ms\n")
cat("全体平均:", round(mean(d$duration), 1), "ms\n")

means_df <- tibble(condition = c("通常", "明瞭"),
                    mean_dur = c(mean(d$duration[d$condition == "通常"]),
                                 mean(d$duration[d$condition == "明瞭"]))) |>
  mutate(condition = factor(condition, levels = c("通常", "明瞭")))
grand_mean <- mean(d$duration)

# grand_mean(234.1)と通常条件の平均(223.1)は11msしか離れておらず、
# 各ラベルをその線の真上/真下に小さくずらすだけでは2行ラベル同士が
# 重なってしまう。そこで、和対比の切片ラベルは線より十分上、処理コー
# ディングの切片ラベルは線より十分下に離し、かつ棒の外側(左右)に
# 置くことで、線・棒・ラベルのどれとも重ならないようにする。
mean_normal <- means_df$mean_dur[means_df$condition == "通常"]

p <- ggplot(means_df, aes(condition, mean_dur, group = 1)) +
  geom_line(colour = "grey55", linewidth = 0.55) +
  geom_point(size = 4, colour = "#0072B2") +
  geom_text(aes(label = sprintf("%.1f ms", mean_dur)), nudge_y = 3.3,
            size = 3.4, colour = "grey15") +
  geom_hline(yintercept = grand_mean, linetype = 2, colour = "#A8681F") +
  geom_hline(yintercept = mean_normal, linetype = 3, colour = "grey40") +
  annotate("label", x = 1.72, y = grand_mean + 1.7,
           label = "和対比の切片 = 全体平均", hjust = 0,
           size = 3.1, colour = "#A8681F", fill = "white", linewidth = 0) +
  annotate("label", x = 1.28, y = mean_normal - 1.7,
           label = "処理コーディングの切片 = 参照水準の平均", hjust = 0,
           size = 3.1, colour = "grey25", fill = "white", linewidth = 0) +
  coord_cartesian(ylim = c(213, 253), clip = "off") +
  labs(x = NULL, y = "持続時間 (ms)",
       caption = "シミュレーションデータ。scripts/R/ch10_worked_example.R より生成。")
save_fig(p, "fig10-01-coding-comparison", chapter = "ch10", width = 6.5, height = 4.2)
cat("完了。図は assets/figures/ch10/ に保存した。\n")


## `ch10_model_comparison.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

「モデル比較と結果の報告」節の作例。交互作用の有無が異なる2つの
混合効果モデルを、最尤法(ML)で推定し、尤度比検定によって比較する。
scripts/R/sim_measures.R が生成する架空データ(measures.csv)を用いる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

measures_path <- here::here("data", "processed", "measures.csv")
if (!file.exists(measures_path)) source(here::here("scripts", "R", "sim_measures.R"))
d <- read_csv(measures_path, show_col_types = FALSE) |>
  mutate(logfreq_c = as.numeric(scale(log(frequency), scale = FALSE)),
         condition = factor(condition))
contrasts(d$condition) <- contr.sum(2)

# 交互作用を含むモデルと含まないモデルを、いずれもML推定で当てはめる
m_full <- lmer(duration ~ condition * logfreq_c + (1 + condition | speaker) + (1 | item),
               data = d, REML = FALSE)
m_reduced <- lmer(duration ~ condition + logfreq_c + (1 + condition | speaker) + (1 | item),
                  data = d, REML = FALSE)

comparison <- anova(m_reduced, m_full)
print(comparison)

chisq <- comparison$Chisq[2]
df    <- comparison$Df[2]
p_val <- comparison$`Pr(>Chisq)`[2]
cat("\n尤度比検定: χ²(", df, ") =", round(chisq, 2), ", p =",
    format.pval(p_val, digits = 3), "\n")


## `ch10_effect_size.R`

対象章: 第10章「統計的推論・回帰分析・混合効果モデル」

「効果量・区間推定・検定力」節の作例。条件間の平均差(ミリ秒)を、
標準化された効果量(Cohen's d)に変換する具体的な数値例。
scripts/R/sim_measures.R が生成する架空データを用いる。

このデータは話者20名×項目30語の交差デザインであり、各話者・各項目が
両条件に反復して現れる(対応のあるデータ)。単純に2条件をプールした
SDで割ると、話者間・項目間のばらつきまで「ノイズ」に含めてしまい、
対応のあるデザインの情報を無視した値になる(Westfall, Kenny, & Judd 2014)。
対応のある効果量(Cohen's d_z、話者×項目ペアごとの差のSDで標準化)を
あわせて計算し、両者の違いを示す。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

measures_path <- here::here("data", "processed", "measures.csv")
if (!file.exists(measures_path)) source(here::here("scripts", "R", "sim_measures.R"))
d <- read_csv(measures_path, show_col_types = FALSE)

by_cond <- d |> group_by(condition) |> summarize(mean_dur = mean(duration), sd_dur = sd(duration), n = n())
print(by_cond)

mean_diff <- diff(by_cond$mean_dur)
pooled_sd <- sqrt(mean(by_cond$sd_dur^2))
cohens_d_naive <- mean_diff / pooled_sd

cat("\n平均差:", round(mean_diff, 1), "ms\n")
cat("(参考、対応を無視した)プールされたSD:", round(pooled_sd, 1), "ms\n")
cat("(参考、対応を無視した)Cohen's d:", round(cohens_d_naive, 2), "\n\n")

wide <- d |>
  select(speaker, item, condition, duration) |>
  pivot_wider(names_from = condition, values_from = duration)
pair_diff <- wide[["明瞭"]] - wide[["通常"]]
cohens_dz <- mean(pair_diff) / sd(pair_diff)

cat("話者×項目ペアごとの差の平均:", round(mean(pair_diff), 1), "ms\n")
cat("差のSD:", round(sd(pair_diff), 1), "ms\n")
cat("対応のある効果量 Cohen's d_z:", round(cohens_dz, 2), "\n")


# 第11章 範疇知覚と手がかりの重みづけ

## `sim_identification_data.R`

対象章: 第8章「知覚実験——刺激作成とPsychoPy」・第11章「範疇知覚と手がかりの重みづけ」

目的:
  実データを収録するまでの間、本書の作例（第11章「発展課題」直前の
  通しの例）で使う同定課題データを、現実的なパラメータでシミュレートする。
  このスクリプトが生成する data/processed/identification_sim.csv は
  **実測データではなくシミュレーションである**ことを、本文・ファイル名の
  両方で明示する。

刺激設計（第8章 §刺激連続体の合成 と対応）:
  scripts/praat/make_vot_continuum.praat と同じ設計思想で、
  VOT 0ms から 80ms まで 10ms 刻みの9段階の連続体を想定する。

個人差のモデル化（第11章 §重みの推定方法と個人差 と対応）:
  各参加者は、母集団の平均境界・平均傾きのまわりに個人差を持つ
  （境界の個人差 SD=8ms、傾きの個人差はロジット尺度で変動）。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_subjects <- 24
steps      <- seq(0, 80, by = 10)          # 9段階（第8章の刺激連続体と対応）
n_reps     <- 12                            # 1参加者・1ステップあたりの繰り返し数

pop_boundary_mean <- 32     # 母集団平均境界（VOT ms）
pop_boundary_sd    <- 8      # 境界の参加者間SD
pop_slope_mean     <- 0.22   # 母集団平均傾き（ロジット/ms）
pop_slope_sd        <- 0.07  # 傾きの参加者間SD（負の値にならないようlogでモデル化）

subj <- tibble(
  subject  = sprintf("s%02d", 1:n_subjects),
  boundary = rnorm(n_subjects, pop_boundary_mean, pop_boundary_sd),
  slope    = exp(rnorm(n_subjects, log(pop_slope_mean), pop_slope_sd / pop_slope_mean))
)

d <- subj |>
  tidyr::crossing(step = steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p = slope * (step - boundary),
    p_resp  = plogis(logit_p),
    response_bin = rbinom(n(), 1, p_resp),
    response = if_else(response_bin == 1, "p", "b")
  ) |>
  select(subject, step, rep, response)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "identification_sim.csv"))

cat("生成したシミュレーションデータ:", nrow(d), "行 /", n_subjects, "名 x",
    length(steps), "ステップ x", n_reps, "反復\n")
cat("母集団パラメータ: 境界平均", pop_boundary_mean, "ms (SD", pop_boundary_sd, "),",
    "傾き平均", pop_slope_mean, "(幾何SD要因", round(exp(pop_slope_sd/pop_slope_mean),2), ")\n")


## `ch11_worked_example.R`

対象章: 第11章「範疇知覚と手がかりの重みづけ」通しの作例

本スクリプトは、第8章で作成したVOT連続体（9段階、0-80ms）に対する
同定課題の反応を、シミュレーションデータ（scripts/R/sim_identification_data.R
が生成する data/processed/identification_sim.csv）に対して実行する。
**実測データではない。** 実データが得られ次第、同じコードをそのまま適用できる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

sim_path <- here::here("data", "processed", "identification_sim.csv")
if (!file.exists(sim_path)) {
  source(here::here("scripts", "R", "sim_identification_data.R"))
}

d <- read_csv(sim_path, show_col_types = FALSE) |>
  mutate(
    subject  = factor(subject),
    response = as.integer(response == "p"),
    step_c   = as.numeric(scale(step, scale = FALSE))
  )

m <- glmer(response ~ step_c + (1 + step_c | subject),
           data = d, family = binomial,
           control = glmerControl(optimizer = "bobyqa",
                                  optCtrl = list(maxfun = 2e5)))

fx <- fixef(m)
step_mean <- mean(d$step)
boundary_pop <- step_mean - fx["(Intercept)"] / fx["step_c"]

co <- coef(m)$subject
per_subj <- tibble(
  subject  = rownames(co),
  slope    = co[, "step_c"],
  boundary = step_mean - co[, "(Intercept)"] / co[, "step_c"]
)

cat("=== 固定効果（母集団平均） ===\n")
print(summary(m)$coefficients)
cat("\n母集団平均のカテゴリー境界（50%点）:", round(boundary_pop, 1), "ms\n")
cat("母集団平均の傾き（ロジット/ms）:", round(fx["step_c"], 3), "\n\n")

cat("=== 参加者ごとの境界・傾き（要約） ===\n")
print(summary(per_subj[c("boundary", "slope")]))
cat("境界の参加者間SD:", round(sd(per_subj$boundary), 1), "ms\n")
cat("傾きの参加者間SD:", round(sd(per_subj$slope), 3), "\n\n")

isSingular_flag <- isSingular(m)
cat("特異適合か:", isSingular_flag, "\n")

# 図: 観測された反応割合 + モデルの当てはめ曲線
obs <- d |>
  group_by(step) |>
  summarise(prop = mean(response), n = n(), .groups = "drop")

curve_df <- tibble(step = seq(min(d$step), max(d$step), length.out = 200)) |>
  mutate(prop = plogis(fx["(Intercept)"] + fx["step_c"] * (step - step_mean)))

p <- ggplot(obs, aes(step, prop)) +
  geom_line(data = curve_df, linewidth = 0.9, colour = "#34518A") +
  geom_point(aes(size = n), colour = "#1B2333") +
  geom_vline(xintercept = as.numeric(boundary_pop), linetype = 2, colour = "#A8681F") +
  scale_y_continuous(labels = scales::percent, limits = c(0, 1)) +
  scale_size_continuous(guide = "none") +
  labs(x = "刺激ステップ (VOT, ms)", y = "「/p/」反応の割合",
       caption = "scripts/R/ch11_worked_example.R より生成（シミュレーションデータ）")
save_fig(p, "fig11-01-identification-curve", chapter = "ch11")

# 図: 参加者ごとの境界・傾きの分布（個人差）
p2 <- ggplot(per_subj, aes(boundary, slope)) +
  geom_point(size = 2, colour = "#34518A", alpha = 0.75) +
  labs(x = "参加者ごとの境界 (VOT, ms)", y = "参加者ごとの傾き（ロジット/ms）",
       caption = "scripts/R/ch11_worked_example.R より生成（シミュレーションデータ）")
save_fig(p2, "fig11-02-boundary-slope-individual", chapter = "ch11")

write_csv(per_subj, here::here("data", "processed", "ch11_per_subject_estimates.csv"))
cat("\n完了。図は assets/figures/ch11/ に、参加者ごとの推定値は data/processed/ch11_per_subject_estimates.csv に保存した。\n")


## `ch11_cue_weighting.R`

対象章: 第11章「範疇知覚と手がかりの重みづけ」通しの作例(2手がかり版)

「重みの推定方法と個人差」節の作例。scripts/R/sim_identification_data.R
の単一手がかり(VOT)の設計を、VOT・F0の立ち上がりという2つの手がかりに
拡張する。実測データではない。各参加者にVOTへの依存度とF0への依存度の
個人差を与え、glmerで交互作用モデルを当てはめたうえで、参加者ごとの
手がかり重み(ランダム傾き)を抽出し、重み空間上の個人差を図示する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260905)

n_subjects <- 24
vot_steps  <- seq(0, 80, by = 20)     # 5段階
f0_steps   <- seq(-40, 40, by = 20)   # 5段階(基準からのHzの立ち上がり差)
n_reps     <- 6

pop_vot_weight <- 0.09    # VOTへの平均的な依存度(ロジット/ms)
pop_f0_weight  <- 0.035   # F0への平均的な依存度(ロジット/Hz)
boundary_vot   <- 40
boundary_f0    <- 0

subj <- tibble(
  subject = sprintf("s%02d", 1:n_subjects),
  # 個人差: VOTへの依存度とF0への依存度を、それぞれ独立な乱数として設定する。
  # 2つの重みのあいだに正・負いずれの相関があるかは、事前に仮定せず、
  # 推定モデルの結果から確認する。
  vot_weight = pmax(rnorm(n_subjects, pop_vot_weight, 0.03), 0.01),
  f0_weight  = pmax(rnorm(n_subjects, pop_f0_weight, 0.013), 0.002)
)

d <- subj |>
  tidyr::crossing(vot = vot_steps) |>
  tidyr::crossing(f0 = f0_steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p  = vot_weight * (vot - boundary_vot) + f0_weight * (f0 - boundary_f0),
    p_resp   = plogis(logit_p),
    response = rbinom(n(), 1, p_resp)
  ) |>
  select(subject, vot, f0, rep, response)

d <- d |> mutate(vot_z = as.numeric(scale(vot)),
                  f0_z  = as.numeric(scale(f0)))
vot_sd <- sd(d$vot)
f0_sd  <- sd(d$f0)

m_cue <- glmer(response ~ vot_z * f0_z + (0 + vot_z + f0_z || subject),
               data = d, family = binomial,
               control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))

print(summary(m_cue)$coefficients)

# 収束警告(max|grad|)が出るため、別の最適化アルゴリズムで固定効果が変わらないかを確かめる
m_alt <- update(m_cue, control = glmerControl(optimizer = "nloptwrap",
                                              optCtrl = list(maxeval = 2e5)))
cat("\n最適化アルゴリズムを変えたときの固定効果の最大の差:",
    signif(max(abs(fixef(m_cue) - fixef(m_alt))), 3), "\n")
cat("対数尤度の差:", signif(as.numeric(logLik(m_cue)) - as.numeric(logLik(m_alt)), 3), "\n")

co <- coef(m_cue)$subject
per_subj <- tibble(
  subject       = rownames(co),
  vot_weight_est = co[, "vot_z"] / vot_sd,
  f0_weight_est  = co[, "f0_z"] / f0_sd
)

comparison <- per_subj |>
  left_join(subj |> select(subject, vot_weight, f0_weight), by = "subject")

r_val <- cor(per_subj$vot_weight_est, per_subj$f0_weight_est)
r_true <- cor(comparison$vot_weight, comparison$f0_weight)
cat("シミュレーションで与えた真の重みの標本相関: r =", round(r_true, 3), "\n")
cat("\nVOT重みとF0重みの参加者間相関: r =", round(r_val, 3), "\n")
# ランダム効果の相関は || で0に固定している。個人別推定値の標本相関を
# 母集団のランダム傾き相関と誤認しないための教育用設計である。
cat("モデル内のランダム効果相関: 0に固定(||指定)\n")
cat("VOT重みの範囲:", round(range(per_subj$vot_weight_est), 3), "\n")
cat("F0重みの範囲:", round(range(per_subj$f0_weight_est), 3), "\n")

p_true <- ggplot(comparison, aes(vot_weight, f0_weight)) +
  geom_point(size = 2.5, colour = "#0072B2", alpha = 0.8) +
  geom_smooth(method = "lm", se = TRUE, colour = "#D55E00", linewidth = 0.7,
              formula = y ~ x) +
  labs(title = sprintf("生成時の真値 (r = %.2f)", r_true),
       x = "VOTへの真の重み (ロジット/ms)",
       y = "F0への真の重み (ロジット/Hz)")

p_est <- ggplot(comparison, aes(vot_weight_est, f0_weight_est)) +
  geom_point(size = 2.5, colour = "#34518A", alpha = 0.8) +
  geom_smooth(method = "lm", se = TRUE, colour = "#A8681F", linewidth = 0.7, formula = y ~ x) +
  labs(title = sprintf("混合モデルによる推定値 (r = %.2f)", r_val),
       x = "VOTへの推定重み (ロジット/ms)",
       y = "F0への推定重み (ロジット/Hz)")

library(patchwork)
p <- p_true + p_est
save_fig(p, "fig11-03-cue-weighting-space", chapter = "ch11", width = 9, height = 4)
cat("完了。図はassets/figures/ch11/に保存した。\n")


# 第12章 韻律研究

## `ch12_worked_example.R`

対象章: 第12章「韻律研究」通しの作例

図12.1: X-JToBIラベルを重ねたF0曲線（模式図）。
  実際の発話の音響測定ではなく、東京方言のアクセント核(H*+L)と
  音韻句境界音調(%L, H-, L%)がF0曲線のどの形状的特徴と対応するかを
  示すための、理想化された模式的な曲線である。

図12.2: Hz尺度と半音(セミトーン)・z得点正規化の比較。
  声域の異なる2名の仮想話者について、同じ形状のF0パターンを
  異なる声域に乗せてシミュレートし、正規化の有無で話者間の差が
  どう変わって見えるかを対比する。いずれも実測データではない。
--- 図12.1: 模式的なF0曲線とX-JToBIラベル -----------------------------

語頭(%L)からアクセント核(H*+L)までは、単調な立ち上がり1本の山だけにする。
(以前の版は「語頭付近の緩やかな立ち上がり」を独立したガウス項として
加算していたため、意図せず2つ目の無ラベルのピークができてしまい、
「図中のすべての形状的特徴がラベルと対応する」という本図の狙いに反していた。
monoH.FC補間で単調な区間だけを作ることで、この問題を避ける。)

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

rise  <- splinefun(c(0, 25, 45), c(130, 150, 168), method = "monoH.FC")
fall  <- splinefun(c(45, 55, 60), c(168, 118, 114), method = "monoH.FC")

t <- seq(0, 100, length.out = 400)
f0_schematic <- ifelse(
  t <= 45, rise(t),
  ifelse(t <= 60, fall(t),
         114 + 10 * exp(-((t - 75) / 10)^2) - 0.02 * (t - 60))  # H-の緩やかな山 + 句末に向けた微下降
)

schematic_df <- tibble(t = t, f0 = f0_schematic)
labels_df <- tibble(
  t     = c(2, 45, 60, 75),
  f0    = c(132, 168, 114, 124),
  label = c("%L", "H*+L", "L", "H-"),
  vjust = c(1.6, -0.6, 1.6, -0.6)
)

p1 <- ggplot(schematic_df, aes(t, f0)) +
  geom_line(linewidth = 1, colour = "#34518A") +
  geom_text(data = labels_df, aes(t, f0, label = label, vjust = vjust),
            size = 4, fontface = "bold", colour = "#A8681F") +
  scale_x_continuous(limits = c(-2, 100)) +
  scale_y_continuous(limits = c(NA, 178)) +
  labs(x = "発話内の相対時間 (%)", y = "F0 (Hz、模式値)",
       caption = "模式図（実測データではない）。scripts/R/ch12_worked_example.R より生成。")
save_fig(p1, "fig12-01-tobi-schematic", chapter = "ch12")

## --- 図12.2: Hz・半音・z得点の比較 --------------------------------------

n_time <- 50
shape <- plogis(seq(-4, 4, length.out = n_time)) - 0.5   # 共通の形状(立ち上がりパターン)

# 話者A: 声域が低い(100-160Hz)、話者B: 声域が高い(190-300Hz)
speakerA <- 130 + shape * 30
speakerB <- 245 + shape * 55

d_hz <- tibble(
  time     = rep(seq(0, 100, length.out = n_time), 2),
  speaker  = rep(c("話者A", "話者B"), each = n_time),
  hz       = c(speakerA, speakerB)
) |>
  group_by(speaker) |>
  mutate(
    semitone = 12 * log2(hz / mean(hz)),
    z_score  = as.numeric(scale(hz))
  ) |>
  ungroup()

d_long <- d_hz |>
  pivot_longer(cols = c(hz, semitone, z_score), names_to = "scale", values_to = "value") |>
  mutate(scale = factor(scale, levels = c("hz", "semitone", "z_score"),
                         labels = c("生のHz", "半音(セミトーン)", "z得点")))

p2 <- ggplot(d_long, aes(time, value, colour = speaker, linetype = speaker)) +
  geom_line(linewidth = 0.9) +
  facet_wrap(~scale, scales = "free_y", nrow = 1) +
  scale_colour_manual(values = c("話者A" = "#34518A", "話者B" = "#A8681F")) +
  scale_linetype_manual(values = c("話者A" = "solid", "話者B" = "22")) +
  labs(x = "発話内の相対時間 (%)", y = NULL, colour = NULL, linetype = NULL,
       caption = "シミュレーションデータ。scripts/R/ch12_worked_example.R より生成。")
save_fig(p2, "fig12-02-normalization-comparison", chapter = "ch12", width = 8.5, height = 3.2)

range_hz       <- diff(range(c(mean(speakerA), mean(speakerB))))
range_semitone <- 12 * log2(mean(speakerB) / mean(speakerA))
cat("話者A平均F0:", round(mean(speakerA), 1), "Hz / 話者B平均F0:", round(mean(speakerB), 1), "Hz\n")
cat("平均F0の話者間差(Hz):", round(range_hz, 1), "Hz\n")
cat("同じ差を半音に換算すると:", round(range_semitone, 2), "semitones\n")
cat("完了。図は assets/figures/ch12/ に保存した。\n")


## `ch12_annotator_agreement.R`

対象章: 第12章「韻律研究」

「ラベリングの信頼性の評価」節の作例。2名のアノテーターが
同じ発話に句境界の強弱(強い境界/弱い境界)をラベル付けしたと想定し、
単純な一致率とCohenのカッパ係数がどう異なるかを、架空のデータで
確認する。境界の大部分が「弱い」に偏っている(不均衡な分布)場合、
単純な一致率が偶然の一致によって高く見えてしまうことを示す。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_tokens <- 300
p_strong <- 0.15   # 真の境界のうち「強い境界」である割合(不均衡な分布)

true_label <- rbinom(n_tokens, 1, p_strong)

# 各アノテーターは真のラベルをある程度正確に判断するが、一定確率で誤る
make_annotator <- function(true, error_rate) {
  flip <- rbinom(length(true), 1, error_rate)
  ifelse(flip == 1, 1 - true, true)
}

annotator_A <- make_annotator(true_label, 0.12)
annotator_B <- make_annotator(true_label, 0.15)

tab <- table(annotator_A, annotator_B)
print(tab)

percent_agreement <- sum(diag(tab)) / sum(tab)

# Cohenのカッパ係数を手計算する(偶然の一致率を差し引く)
p_o <- percent_agreement
marg_A <- rowSums(tab) / sum(tab)
marg_B <- colSums(tab) / sum(tab)
p_e <- sum(marg_A * marg_B)
kappa <- (p_o - p_e) / (1 - p_e)

cat("単純な一致率:", round(percent_agreement * 100, 1), "%\n")
cat("偶然の一致率(期待値):", round(p_e * 100, 1), "%\n")
cat("Cohenのカッパ係数:", round(kappa, 3), "\n")

# 比較のため、境界がほぼ半々に分布する場合(バランスの取れた分布)も試す
true_label_balanced <- rbinom(n_tokens, 1, 0.5)
annotator_A2 <- make_annotator(true_label_balanced, 0.12)
annotator_B2 <- make_annotator(true_label_balanced, 0.15)
tab2 <- table(annotator_A2, annotator_B2)
p_o2 <- sum(diag(tab2)) / sum(tab2)
marg_A2 <- rowSums(tab2) / sum(tab2)
marg_B2 <- colSums(tab2) / sum(tab2)
p_e2 <- sum(marg_A2 * marg_B2)
kappa2 <- (p_o2 - p_e2) / (1 - p_e2)

cat("\n[比較: 境界がほぼ半々に分布する場合]\n")
cat("単純な一致率:", round(p_o2 * 100, 1), "%\n")
cat("偶然の一致率(期待値):", round(p_e2 * 100, 1), "%\n")
cat("Cohenのカッパ係数:", round(kappa2, 3), "\n")


## `ch12_time_normalization.R`

対象章: 第12章「韻律研究」

「F0の測定・正規化・時間正規化」節の作例。話速の異なる2名の話者が、
言語的には同じ相対位置(発話の70%地点)でF0ピークに達するとき、
秒単位の絶対時間で比較すると、ピーク時刻がずれて見えることを確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

speaker_a_duration <- 0.60   # 秒、話速が速い話者
speaker_b_duration <- 1.00   # 秒、話速が遅い話者
relative_peak_position <- 0.70   # 発話の70%地点でピークに達する(言語的には共通)

peak_time_a_sec <- speaker_a_duration * relative_peak_position
peak_time_b_sec <- speaker_b_duration * relative_peak_position

cat("話者A(発話長0.60秒)のピーク時刻(絶対時間): ", peak_time_a_sec, "秒\n")
cat("話者B(発話長1.00秒)のピーク時刻(絶対時間): ", peak_time_b_sec, "秒\n")
cat("絶対時間で見た2話者のピーク時刻差: ", round(peak_time_b_sec - peak_time_a_sec, 2), "秒\n")
cat("相対時間(0-100%)で見た2話者のピーク位置差: ",
    round((peak_time_b_sec / speaker_b_duration - peak_time_a_sec / speaker_a_duration) * 100, 1),
    "%\n")


## `ch12_measurement_dilution.R`

対象章: 第12章「韻律研究」

「境界と卓立の音響的相関」節の作例。境界前長音化が句末音節の韻
(rhyme)にのみ局所的に生じる場合、測定区間を韻だけに絞るか、
単語全体(複数音節)を平均するかで、検出される効果量がどう変わるかを、
架空のデータで確認する。Wightman et al. (1992)の知見を数値で例示する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 100
true_lengthening_ms <- 40   # 境界直前の韻だけに生じる真の伸長効果

# 句末語は2音節(音節1+音節2)からなり、韻の伸長は音節2にのみ生じるとする
syll1_boundary <- rnorm(n_tokens, 120, 15)
syll2_rhyme_boundary <- rnorm(n_tokens, 150, 15) + true_lengthening_ms
syll1_nonboundary <- rnorm(n_tokens, 120, 15)
syll2_rhyme_nonboundary <- rnorm(n_tokens, 150, 15)

# 測定1: 韻だけに絞った測定(音節2のみ)。伸長率は韻自身の持続時間に対する割合。
effect_rhyme_ms <- mean(syll2_rhyme_boundary) - mean(syll2_rhyme_nonboundary)
pct_rhyme_only <- effect_rhyme_ms / mean(syll2_rhyme_nonboundary) * 100

# 測定2: 単語全体(2音節の合計)で平均した測定。伸長率は単語全体の持続時間に対する割合。
word_total_boundary <- syll1_boundary + syll2_rhyme_boundary
word_total_nonboundary <- syll1_nonboundary + syll2_rhyme_nonboundary
effect_word_ms <- mean(word_total_boundary) - mean(word_total_nonboundary)
pct_word_total <- effect_word_ms / mean(word_total_nonboundary) * 100

cat("真の伸長効果(韻のみに生じる):", true_lengthening_ms, "ms\n\n")
cat("韻だけに絞った測定: 絶対差 =", round(effect_rhyme_ms, 1),
    "ms, 伸長率 =", round(pct_rhyme_only, 1), "%\n")
cat("単語全体(2音節合計)で平均した測定: 絶対差 =", round(effect_word_ms, 1),
    "ms, 伸長率 =", round(pct_word_total, 1), "%\n")
cat("\n伸長率で報告する場合、単語全体を測定区間にすると、真の伸長率のおよそ",
    round(pct_word_total / pct_rhyme_only * 100, 0), "%しか検出されない\n")


## `ch12_prominence_indicators.R`

対象章: 第12章「韻律研究」

「境界と卓立の音響的相関」節の作例。プロミネンスの複数の音響的指標
(F0ピーク・持続時間・強度)が、部分的に独立しながらも相関して現れる
という想定のもとで、単一の指標だけで卓立の有無を判定した場合と、
複数指標を組み合わせて判定した場合とで、真の卓立をどれだけ正しく
検出できるかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_tokens <- 200
true_prominent <- rbinom(n_tokens, 1, 0.5)

# 各指標は真の卓立を部分的にしか反映せず、指標ごとに異なるノイズ源を持つ
f0_peak   <- 200 + true_prominent * 30 + rnorm(n_tokens, 0, 25)
duration  <- 120 + true_prominent * 15 + rnorm(n_tokens, 0, 20)
intensity <- 65  + true_prominent * 3  + rnorm(n_tokens, 0, 4)

judge_single <- function(x) as.integer(x > median(x))
judge_f0        <- judge_single(f0_peak)
judge_duration  <- judge_single(duration)
judge_intensity <- judge_single(intensity)

# 複合指標: 3つの指標をz得点化して平均し、その中央値で二値化する
z <- function(x) (x - mean(x)) / sd(x)
composite <- (z(f0_peak) + z(duration) + z(intensity)) / 3
judge_composite <- judge_single(composite)

accuracy <- function(judged) mean(judged == true_prominent)

cat("F0ピークのみでの判定精度:", round(accuracy(judge_f0) * 100, 1), "%\n")
cat("持続時間のみでの判定精度:", round(accuracy(judge_duration) * 100, 1), "%\n")
cat("強度のみでの判定精度:", round(accuracy(judge_intensity) * 100, 1), "%\n")
cat("3指標の複合スコアでの判定精度:", round(accuracy(judge_composite) * 100, 1), "%\n")


# 第13章 音韻変異とコーパス音韻論

## `ch13_worked_example.R`

対象章: 第13章「音韻変異とコーパス音韻論」通しの作例

「予測可能性・頻度・レマの効果」節で述べた、ジュラフスキーら[-@jurafsky2002]
およびガール[-@gahl2008]の知見(頻度・予測可能性が高い語ほど短く発音される)
を、架空のコーパスデータで再現する。実測データではない。

データ生成の設計:
  50語(レマ)を、対数頻度が一様に分布するように設定し、各レマについて
  20名の話者×平均4トークンの出現を生成する。真の効果として、
  log(頻度)が1単位上がるごとに持続時間が対数尺度で一定割合短くなる
  ように設定し(頻度効果)、文脈的な予測可能性(0-1)が高いほど
  さらに短縮するように設定する(予測可能性効果)。話者ごとの
  発話速度の個人差もランダム効果として加える。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

n_lemma    <- 50
n_speakers <- 20

lemmas <- tibble(
  lemma      = sprintf("lemma%02d", 1:n_lemma),
  log_freq   = runif(n_lemma, 0, 8),                 # 対数頻度(コーパス内トークン数の対数相当)
  lemma_re   = rnorm(n_lemma, 0, 0.05)                # レマ固有のランダム切片(対数尺度)
)

speakers <- tibble(
  speaker    = sprintf("sp%02d", 1:n_speakers),
  speaker_re = rnorm(n_speakers, 0, 0.08)             # 話者固有の発話速度差(対数尺度)
)

true_freq_effect <- -0.035   # log(頻度)1単位あたりの対数持続時間の変化
true_pred_effect <- -0.25    # 予測可能性(0-1)1単位あたりの対数持続時間の変化
base_log_dur      <- log(220) # 基準となる持続時間(約220ms)

d <- lemmas |>
  tidyr::crossing(speakers) |>
  rowwise() |>
  mutate(n_tok = rpois(1, 4) + 1) |>
  ungroup() |>
  tidyr::uncount(n_tok) |>
  mutate(
    predictability = runif(n(), 0, 1),
    log_dur_mean = base_log_dur +
      true_freq_effect * (log_freq - mean(log_freq)) +
      true_pred_effect * (predictability - 0.5) +
      lemma_re + speaker_re,
    log_duration = log_dur_mean + rnorm(n(), 0, 0.12),
    duration_ms  = exp(log_duration)
  )

cat("生成したトークン数:", nrow(d), " / レマ数:", n_lemma, " / 話者数:", n_speakers, "\n\n")

m <- lmer(log(duration_ms) ~ scale(log_freq, scale = FALSE) + predictability +
            (1 | speaker) + (1 | lemma), data = d)
print(summary(m))

fx <- fixef(m)
pct_per_logfreq <- (exp(fx["scale(log_freq, scale = FALSE)"]) - 1) * 100
pct_per_pred    <- (exp(fx["predictability"]) - 1) * 100

cat("\n対数頻度1単位あたりの持続時間の変化:", round(pct_per_logfreq, 2), "%\n")
cat("予測可能性が0から1に上がるときの持続時間の変化:", round(pct_per_pred, 2), "%\n")

p <- d |>
  mutate(freq_bin = cut(log_freq, breaks = 6)) |>
  group_by(freq_bin) |>
  summarise(mean_dur = mean(duration_ms), mean_logfreq = mean(log_freq), n = n(), .groups = "drop")

p1 <- ggplot(d, aes(log_freq, duration_ms)) +
  geom_point(alpha = 0.15, size = 0.8, colour = "#34518A") +
  geom_smooth(method = "lm", colour = "#A8681F", linewidth = 1, se = TRUE) +
  labs(x = "レマの対数頻度", y = "持続時間 (ms)",
       caption = "シミュレーションデータ。scripts/R/ch13_worked_example.R より生成。")
save_fig(p1, "fig13-01-frequency-duration", chapter = "ch13")

write_csv(d, here::here("data", "processed", "ch13_lemma_duration_sim.csv"))
cat("\n完了。図は assets/figures/ch13/ に保存した。\n")


## `ch13_large_n_problem.R`

対象章: 第13章「音韻変異とコーパス音韻論」

「大N問題」節の作例。理論的には無視できるほど小さな効果量でも、
サンプルサイズが極端に大きければ統計的に有意と判定されてしまうことを、
架空のデータで実際に確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(10)

trivial_effect_ms <- 0.5   # 理論的には無視できる、きわめて小さな効果量
sd_duration <- 30

for (n in c(100, 5000, 200000)) {
  a <- rnorm(n, 150, sd_duration)
  b <- rnorm(n, 150 + trivial_effect_ms, sd_duration)
  test <- t.test(b, a)
  cat("N =", n, "(各群): 観測された差 =", round(diff(test$estimate) * -1, 3),
      "ms, p =", format.pval(test$p.value, digits = 3), "\n")
}


## `ch13_speaker_bootstrap.R`

対象章: 第13章「音韻変異とコーパス音韻論」

「大N問題」節の作例。トークン単位でリサンプリングした場合と、話者
単位でリサンプリングした場合とで、効果量の不確実性(標準誤差)の
推定がどう異なるかを、話者間相関を持つ架空のコーパスデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n_speakers <- 15
n_tokens_per_speaker <- 200

speakers <- tibble(speaker = sprintf("sp%02d", 1:n_speakers),
                    spk_re = rnorm(n_speakers, 0, 15))   # 話者ごとの安定した個人差

d <- speakers |>
  tidyr::crossing(token = 1:n_tokens_per_speaker) |>
  mutate(duration = 150 + spk_re + rnorm(n(), 0, 8))

n_boot <- 500

boot_token_level <- map_dbl(1:n_boot, function(i) {
  resampled <- d |> slice_sample(n = nrow(d), replace = TRUE)
  mean(resampled$duration)
})

boot_speaker_level <- map_dbl(1:n_boot, function(i) {
  resampled_speakers <- sample(speakers$speaker, n_speakers, replace = TRUE)
  resampled <- d |> filter(speaker %in% resampled_speakers)
  mean(resampled$duration)
})

cat("トークン単位のブートストラップによるSE:", round(sd(boot_token_level), 3), "\n")
cat("話者単位のブートストラップによるSE:", round(sd(boot_speaker_level), 3), "\n")
cat("SEの比率(話者単位/トークン単位):",
    round(sd(boot_speaker_level) / sd(boot_token_level), 1), "倍\n")


# 第14章 音韻習得研究——第二言語を中心に

## `ch14_pam_schematic.R`

対象章: 第14章「音韻習得研究——第二言語を中心に」

PAM/PAM-L2が区別する同化パターンの3類型(Two Category, Single Category,
Uncategorized-Categorized)を、母語カテゴリーの音響空間上に2つの
第二言語音がどう位置づけられるかとして図示する模式図。
実測データではなく、理論的な予測を視覚化するための概念図である。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(patchwork)

native_cats <- tibble(
  cat   = c("母語カテゴリーA", "母語カテゴリーB"),
  lab   = c("母語\nカテゴリーA", "母語\nカテゴリーB"),
  xmin  = c(0.5, 3.5),
  xmax  = c(2.0, 5.0)
)

make_panel <- function(pattern, l2_sounds, title, note) {
  # 白黒印刷で母語カテゴリーA/Bが区別できるよう、濃い灰色(A)と薄い灰色(B)に
  # し、境界線も付ける。L2音は黒の三角で示し、ラベルは上下にずらして重ねない。
  l2_sounds <- l2_sounds |> mutate(ly = c(0.80, 0.62))
  ggplot() +
    geom_rect(data = native_cats, aes(xmin = xmin, xmax = xmax, ymin = 0, ymax = 1, fill = cat),
              colour = "grey20", linewidth = 0.4) +
    geom_point(data = l2_sounds, aes(x = x, y = 0.45), size = 4.5, shape = 17, colour = "black") +
    geom_segment(data = l2_sounds, aes(x = x, xend = x, y = 0.45, yend = ly - 0.05),
                 colour = "black", linewidth = 0.3) +
    geom_label(data = l2_sounds, aes(x = x, y = ly, label = label), size = 4.2,
               fontface = "bold", colour = "black", fill = "white", linewidth = 0.3) +
    geom_text(data = native_cats, aes(x = (xmin + xmax) / 2, y = 0.14, label = lab), size = 2.9, lineheight = 0.95,
              fontface = "bold", colour = c("white", "black")) +
    scale_fill_manual(values = c("母語カテゴリーA" = "#4D4D4D", "母語カテゴリーB" = "#C8C8C8"), guide = "none") +
    scale_x_continuous(limits = c(0, 5.5), breaks = NULL, name = NULL) +
    scale_y_continuous(limits = c(0, 1), breaks = NULL, name = NULL) +
    labs(title = title, caption = note) +
    theme(panel.grid = element_blank(), plot.title = element_text(size = 12, face = "bold"),
          plot.caption = element_text(size = 10, colour = "black", hjust = 0, lineheight = 1.05))
}

p_tc <- make_panel(
  "TC",
  tibble(x = c(1.2, 4.2), label = c("L2音1", "L2音2")),
  "Two Category (TC)",
  "2つのL2音が別々の母語カテゴリーに同化\n→ 弁別は容易と予測"
)

p_sc <- make_panel(
  "SC",
  tibble(x = c(1.0, 1.6), label = c("L2音1", "L2音2")),
  "Single Category (SC)",
  "2つのL2音が同じ母語カテゴリーに\n強く同化(良さの差は小さい)\n→ 弁別は困難と予測"
)

p_uc <- make_panel(
  "UC",
  tibble(x = c(2.7, 4.2), label = c("L2音1", "L2音2")),
  "Uncategorized-Categorized (UC)",
  "1つは範疇間の隙間に非範疇化、\nもう1つは明確に同化\n→ 弁別可能性は他要因に依存"
)

p <- p_tc + p_sc + p_uc + plot_layout(nrow = 1)
ggsave(here::here("assets", "figures", "ch14", "fig14-01-pam-assimilation.png"),
       p, width = 9, height = 3.9, dpi = 320)
ggsave(here::here("assets", "figures", "ch14", "fig14-01-pam-assimilation.pdf"),
       p, width = 9, height = 3.9, device = cairo_pdf)
cat("完了。図はassets/figures/ch14/に保存した。\n")


## `ch14_perception_production_correlation.R`

対象章: 第14章「音韻習得研究——第二言語を中心に」

「産出の測定と、知覚と産出の対応」節の作例。知覚成績と産出成績が
真には中程度に相関している(r = 0.6)場合でも、それぞれの測定に
独立な誤差が乗ると、観測される相関係数がどれだけ減衰するかを、
架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 100
true_r <- 0.6

# 2変量正規分布から、真の相関r=0.6を持つ潜在的な「知覚能力」「産出能力」を生成
latent <- MASS::mvrnorm(n, mu = c(0, 0),
                         Sigma = matrix(c(1, true_r, true_r, 1), nrow = 2))
true_perception <- latent[, 1]
true_production <- latent[, 2]

measurement_noise_sd <- 0.8   # 各課題の測定に独立に乗る誤差

observed_perception <- true_perception + rnorm(n, 0, measurement_noise_sd)
observed_production <- true_production + rnorm(n, 0, measurement_noise_sd)

cat("真の潜在能力間の相関(理論値): r =", true_r, "\n")
cat("真の潜在変数どうしの標本相関: r =",
    round(cor(true_perception, true_production), 3), "\n")
cat("測定誤差を含む観測値どうしの相関: r =",
    round(cor(observed_perception, observed_production), 3), "\n")


## `ch14_measurement_error_attenuation.R`

対象章: 第14章「音韻習得研究——第二言語を中心に」

「個人差・経験変数のモデル化」節の作例。自己報告による経験変数
(LOR)に測定誤差(想起バイアス)が含まれる場合、真の経験変数を
使った回帰係数に比べて、自己報告値を使った回帰係数がどれだけ
減衰(過小評価)するかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 150
true_lor_years <- runif(n, 0, 10)          # 真の居住期間(年)
true_effect <- -3                          # LOR1年あたりの知覚困難度の真の減少

# 自己報告値には、想起バイアスによる測定誤差が乗る(標準偏差1.5年)
self_reported_lor <- true_lor_years + rnorm(n, 0, 1.5)

outcome <- 50 + true_effect * true_lor_years + rnorm(n, 0, 8)

m_true <- lm(outcome ~ true_lor_years)
m_self_reported <- lm(outcome ~ self_reported_lor)

cat("真のLORを予測変数とした場合の係数:", round(coef(m_true)[2], 2), "\n")
cat("自己報告LOR(測定誤差あり)を予測変数とした場合の係数:",
    round(coef(m_self_reported)[2], 2), "\n")
cat("減衰率:", round((1 - coef(m_self_reported)[2] / coef(m_true)[2]) * 100, 1), "%\n")


## `ch14_worked_example.R`

対象章: 第14章「音韻習得研究——第二言語を中心に」通しの作例（第11章の同定課題の枠組みを応用）

「発音訓練研究とHVPT」節で述べた、高変動音素訓練の効果(訓練前後の
知覚改善、および新規話者・語彙への般化)を、架空のデータでシミュレートする。
実測データではない。刺激は第11章と同じくVOT連続体を想定するが、
ここでは日本語母語話者による英語/r/-/l/の同定課題を模した設定とする。

設計:
  同一の仮想学習者20名について、訓練前(pre)・訓練後(post)・
  訓練で使わなかった新規話者/語彙での般化テスト(generalization)の
  3条件で同定課題を行わせる。訓練前は境界が不安定(傾きが緩やか)で
  あるのに対し、訓練後は傾きが急になる(範疇知覚がより明確になる)と
  設定し、般化条件では訓練後の改善の一部が保持されるが完全ではない
  ように設定する。

統計的な扱い: 条件ごとに別々のモデルを当てるのではなく、条件を予測変数に
含めた単一のモデル(step_c * condition)を当て、条件間の傾きの差を
交互作用項として直接検定する。傾きの急峻化だけをもって「学習が改善した」
とは判断せず、刺激連続体の両端(範疇的にもっとも明確な刺激)での正答率も
あわせて確認し、境界の位置が正しいカテゴリーに対応しているかを検証する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

n_subjects <- 20
steps      <- seq(0, 80, by = 10)
n_reps     <- 10
true_boundary <- 35   # /r/-/l/の物理的な境界と想定する値(このステップ未満が正しく/r/)

pop <- tibble(
  condition = c("pre", "post", "generalization"),
  slope_mean = c(0.09, 0.24, 0.19),     # 訓練前は緩やか、訓練後は急峻、般化はその中間
  boundary_mean = c(38, 33, 34)
)

subj_re <- tibble(
  subject = sprintf("s%02d", 1:n_subjects),
  slope_re    = rnorm(n_subjects, 0, 0.03),
  boundary_re = rnorm(n_subjects, 0, 5)
)

d <- pop |>
  tidyr::crossing(subj_re) |>
  mutate(slope = pmax(slope_mean + slope_re, 0.02),
         boundary = boundary_mean + boundary_re) |>
  tidyr::crossing(step = steps) |>
  tidyr::crossing(rep = 1:n_reps) |>
  mutate(
    logit_p = slope * (step - boundary),
    p_resp  = plogis(logit_p),
    response = rbinom(n(), 1, p_resp)
  ) |>
  select(subject, condition, step, rep, response) |>
  mutate(
    condition = factor(condition, levels = c("pre", "post", "generalization")),
    step_c    = as.numeric(scale(step, scale = FALSE))
  )

step_mean <- mean(d$step)

# 条件を予測変数に含めた単一モデル。conditionはpreを参照水準とする
# 処理コーディング(第10章参照)なので、step_c:conditionの係数がそのまま
# 「pre条件と比べた傾きの差」の検定になる。
m <- glmer(response ~ step_c * condition + (1 + step_c | subject),
           data = d, family = binomial,
           control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))

cat("=== 単一モデルの固定効果(条件をpre基準として) ===\n")
print(summary(m)$coefficients)

fx <- fixef(m)
slope_pre   <- fx["step_c"]
slope_post  <- fx["step_c"] + fx["step_c:conditionpost"]
slope_gen   <- fx["step_c"] + fx["step_c:conditiongeneralization"]

# post条件をconditionの参照水準に張り替えて再当てはめし、post対generalizationの
# 傾きの差についても、単一モデルの交互作用項として直接検定する
d_relevel <- d |> mutate(condition = relevel(condition, ref = "post"))
m_post_ref <- glmer(response ~ step_c * condition + (1 + step_c | subject),
                     data = d_relevel, family = binomial,
                     control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5)))
coef_post_ref <- summary(m_post_ref)$coefficients

cat("\n=== 傾きの交互作用検定(単一モデル内) ===\n")
cat("post - pre の傾きの差:", round(fx["step_c:conditionpost"], 3),
    " z =", round(summary(m)$coefficients["step_c:conditionpost", "z value"], 2),
    " p =", signif(summary(m)$coefficients["step_c:conditionpost", "Pr(>|z|)"], 3), "\n")
cat("generalization - pre の傾きの差:", round(fx["step_c:conditiongeneralization"], 3),
    " z =", round(summary(m)$coefficients["step_c:conditiongeneralization", "z value"], 2),
    " p =", signif(summary(m)$coefficients["step_c:conditiongeneralization", "Pr(>|z|)"], 3), "\n")
cat("generalization - post の傾きの差:", round(coef_post_ref["step_c:conditiongeneralization", "Estimate"], 3),
    " z =", round(coef_post_ref["step_c:conditiongeneralization", "z value"], 2),
    " p =", signif(coef_post_ref["step_c:conditiongeneralization", "Pr(>|z|)"], 3), "\n")

results <- tibble(
  condition = c("pre", "post", "generalization"),
  slope     = c(slope_pre, slope_post, slope_gen)
) |>
  mutate(
    intercept = c(fx["(Intercept)"],
                  fx["(Intercept)"] + fx["conditionpost"],
                  fx["(Intercept)"] + fx["conditiongeneralization"]),
    boundary  = step_mean - intercept / slope
  )
print(results)

# 傾きの急峻化だけでは「学習が改善した」ことを意味しない。境界が正しい
# カテゴリー(true_boundary)の近くに位置し、連続体の両端で高い正答率が
# 得られているかも、あわせて確認する。
acc_at_ends <- d |>
  filter(step %in% c(min(steps), max(steps))) |>
  group_by(condition, step) |>
  summarise(prop_response = mean(response), .groups = "drop") |>
  mutate(prop_correct = if_else(step < true_boundary, 1 - prop_response, prop_response))

cat("\n=== 刺激連続体両端での正答率(境界の妥当性の確認) ===\n")
print(acc_at_ends |> select(condition, step, prop_correct))

curve_df <- purrr::map_dfr(seq_len(nrow(results)), function(i) {
  tibble(condition = results$condition[i],
         step = seq(min(steps), max(steps), length.out = 200)) |>
    mutate(prop = plogis(results$intercept[i] + results$slope[i] * (step - step_mean)))
})

cond_labels <- c(pre = "訓練前", post = "訓練後", generalization = "般化テスト")
curve_df <- curve_df |> mutate(condition_ja = factor(cond_labels[as.character(condition)], levels = cond_labels))

p <- ggplot(curve_df, aes(step, prop, colour = condition_ja, linetype = condition_ja)) +
  geom_line(linewidth = 1) +
  scale_colour_manual(values = c("訓練前" = "#8C8C8C", "訓練後" = "#34518A", "般化テスト" = "#A8681F")) +
  scale_linetype_manual(values = c("訓練前" = "solid", "訓練後" = "22", "般化テスト" = "42")) +
  scale_y_continuous(labels = scales::percent, limits = c(0, 1)) +
  labs(x = "刺激ステップ", y = "「学習対象カテゴリー」反応の割合", colour = NULL, linetype = NULL,
       caption = "シミュレーションデータ。scripts/R/ch14_worked_example.R より生成。")
save_fig(p, "fig14-02-hvpt-curves", chapter = "ch14")

cat("\n傾きの変化(訓練前→訓練後):",
    round(results$slope[results$condition == "pre"], 3), "→",
    round(results$slope[results$condition == "post"], 3), "\n")
cat("般化テストでの傾き:", round(results$slope[results$condition == "generalization"], 3),
    "(訓練後の改善のうち",
    round(100 * (results$slope[results$condition == "generalization"] - results$slope[results$condition == "pre"]) /
            (results$slope[results$condition == "post"] - results$slope[results$condition == "pre"]), 0),
    "%を保持)\n")
cat("完了。図は assets/figures/ch14/ に保存した。\n")


## `ch14_three_dimension_dissociation.R`

対象章: 第14章「音韻習得研究——第二言語を中心に」

「accentedness・comprehensibility・intelligibilityの区別」節の作例。
3つの構成概念が部分的にしか相関しない(独立した情報を持つ)場合、
「訛りは強いが理解は容易」「訛りは弱いが理解に労力を要する」といった
話者が、どれだけの割合で実際に存在しうるかを、架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

n <- 200
accentedness <- rnorm(n, 50, 15)
# comprehensibilityはaccentednessと中程度に相関しつつ、独立な要因も持つ
comprehensibility <- 0.5 * accentedness + rnorm(n, 25, 15)

d <- tibble(accentedness, comprehensibility)

cat("accentednessとcomprehensibilityの相関: r =",
    round(cor(accentedness, comprehensibility), 2), "\n")

# 「訛りは強い(上位25%)が理解は容易(下位25%、低いほど理解しやすいと仮定)」話者
strong_accent <- accentedness > quantile(accentedness, 0.75)
easy_comprehend <- comprehensibility < quantile(comprehensibility, 0.25)
n_dissociated <- sum(strong_accent & easy_comprehend)

cat("訛りが強い(上位25%)話者数:", sum(strong_accent), "\n")
cat("そのうち理解が容易(comprehensibility下位25%)な話者数:", n_dissociated,
    "(", round(n_dissociated / sum(strong_accent) * 100, 1), "%)\n")


# 第15章 不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA

## `sim_f0_tracks.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」(第12章のGAMM想定例と対応)

目的:
  scripts/R/30_gamm_f0.R が読み込む data/processed/f0_tracks.csv を、
  東京方言アクセントの核あり語・無核語という想定例に沿ってシミュレートする。
  **実測データではない。** 核あり語はアクセント核の直後(正規化時間の
  中盤)で急激なF0下降(H*+L)を示すのに対し、無核語はなだらかに
  上昇を続けるパターンとして設計する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)
library(tidyverse)

n_speakers   <- 15
n_utt_per    <- 6   # 話者ごとの発話数(条件あたり3発話)
n_timepoints <- 21  # 正規化時間 0 から 1 まで

speaker_mean_f0 <- tibble(
  speaker = sprintf("sp%02d", 1:n_speakers),
  base_f0 = rnorm(n_speakers, 180, 25)
)

time_norm <- seq(0, 1, length.out = n_timepoints)

make_curve <- function(condition, base_f0) {
  if (condition == "核あり") {
    # 立ち上がり(~0.3)でピーク、その直後(H*+L)で急落
    20 * plogis((time_norm - 0.25) / 0.05) - 35 * plogis((time_norm - 0.5) / 0.04) + base_f0
  } else {
    # 無核: なだらかな上昇を保つ
    15 * plogis((time_norm - 0.4) / 0.15) + base_f0
  }
}

d <- speaker_mean_f0 |>
  tidyr::crossing(utt_in_speaker = 1:n_utt_per) |>
  mutate(condition = rep(c("核あり", "無核"), length.out = n()),
         utterance = paste0(speaker, "_u", utt_in_speaker)) |>
  rowwise() |>
  mutate(curve = list(make_curve(condition, base_f0) + rnorm(n_timepoints, 0, 4))) |>
  ungroup() |>
  select(speaker, utterance, condition, curve) |>
  mutate(time_norm = list(time_norm)) |>
  tidyr::unnest(cols = c(curve, time_norm)) |>
  rename(f0 = curve) |>
  select(speaker, utterance, condition, time_norm, f0)

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
write_csv(d, here::here("data", "processed", "f0_tracks.csv"))
cat("生成した行数:", nrow(d), " / 話者数:", n_speakers,
    " / 発話数:", n_distinct(d$utterance), "\n")


## `30_gamm_f0.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」（第12章の問題を受ける）

なぜ GAMM を使うのか:
  F0 曲線を「平均 F0」や「最大値」といった1点に要約すると、
  曲線の形そのものが持つ情報（いつ上がり、いつ下がるか）が失われる。
  韻律理論の予測は多くの場合その形についての予測であり、
  要約統計量では検証にならない。GAMM は時間の関数として F0 を推定し、
  条件間の差が時間軸のどこに生じるかを示せる。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(mgcv)
library(itsadug)

d <- read_csv(here::here("data", "processed", "f0_tracks.csv"),
              show_col_types = FALSE) |>
  mutate(
    speaker   = factor(speaker),
    condition = factor(condition),
    # 話者内で半音正規化する。Hz のままだと話者の声域差が条件効果を覆う。
    f0_st = 12 * log2(f0 / ave(f0, speaker, FUN = function(x) median(x, na.rm = TRUE)))
  ) |>
  arrange(speaker, utterance, time_norm) |>
  group_by(speaker, utterance) |>
  mutate(start_event = row_number() == 1) |>
  ungroup()

# 条件ごとの平滑 + 話者ごとのランダム平滑。
# 時系列の自己相関を無視すると有意差が出すぎるため、rho で補正する。
m0 <- bam(f0_st ~ condition +
            s(time_norm, by = condition, k = 20) +
            s(time_norm, speaker, bs = "fs", m = 1),
          data = d, method = "fREML", discrete = TRUE)

rho <- start_value_rho(m0)
m1 <- bam(f0_st ~ condition +
            s(time_norm, by = condition, k = 20) +
            s(time_norm, speaker, bs = "fs", m = 1),
          data = d, method = "fREML", discrete = TRUE,
          rho = rho, AR.start = d$start_event)

print(summary(m1))
gam.check(m1)   # k が足りているかを確認する

# 条件間の差が「時間軸のどこに」あるかを見る。
# これが GAMM を使う最大の理由である（第12章の問いへの答え）。
# 基本グラフィックスの既定フォントには日本語グリフが無く、軸ラベルが□に
# 化けるため、日本語フォントを明示する(Macはヒラギノ、それ以外はshowtextで
# 登録済みの"notosans")。
fig_path <- here::here("assets", "figures", "ch15", "fig15-02-f0-difference.png")
if (Sys.info()[["sysname"]] == "Darwin") {
  quartzFonts(japanese = quartzFont(rep("Hiragino Sans", 4)))
  png(fig_path, width = 1600, height = 1100, res = 320, type = "quartz", family = "japanese")
  par(family = "japanese")
} else {
  png(fig_path, width = 1600, height = 1100, res = 320, type = "cairo")
  par(family = "notosans")
}
# sim.ci = TRUE: 既定の逐点(pointwise)信頼区間ではなく、時間軸全体を通じた
# 同時(simultaneous)信頼区間を用いる。逐点区間で「有意な区間」を報告すると、
# 多数の時点それぞれで独立に検定したのと同じことになり、多重比較の問題で
# 有意な区間を過大に検出しやすい。
plot_diff(m1, view = "time_norm",
          comp = list(condition = levels(d$condition)[1:2]),
          sim.ci = TRUE,
          main = "", xlab = "正規化時間", ylab = "差 (半音)",
          hide.label = TRUE,   # 右端の英語自動ラベルは消す(区間の定義は本文・キャプションで述べる)
          col.diff = "grey15") # 有意区間の印は赤でなく濃い灰色(白黒印刷でも判別できる)
dev.off()


## `ch15_worked_example.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」通しの作例

「事前分布の選択と感度分析」節で述べた、架空の説明用の数値例
(事後平均0.42、95%信用区間[0.05, 0.79]相当)を、実際に正規分布から
20,000サンプルを描いてROPE([-0.1, 0.1])との重なりを図示する。
ここでの数値そのものは架空の説明用サンプルであり、実データの解析結果ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260904)

# 事後分布の模式図(ベイズ推定の考え方を説明するための、架空の説明用サンプル)
posterior_draws <- rnorm(20000, mean = 0.42, sd = 0.19)
rope <- c(-0.1, 0.1)
pct_in_rope <- mean(posterior_draws > rope[1] & posterior_draws < rope[2]) * 100
ci95 <- quantile(posterior_draws, c(0.025, 0.975))

d <- tibble(beta = posterior_draws)
p <- ggplot(d, aes(beta)) +
  geom_density(fill = "#34518A", alpha = 0.25, colour = "#34518A", linewidth = 0.8) +
  geom_vline(xintercept = rope, linetype = 2, colour = "#A8681F") +
  geom_vline(xintercept = ci95, linetype = 3, colour = "grey30") +
  annotate("rect", xmin = rope[1], xmax = rope[2], ymin = 0, ymax = Inf,
           fill = "#A8681F", alpha = 0.12) +
  labs(x = expression(beta ~ "(効果量、架空の説明用サンプル)"), y = "事後密度",
       caption = paste0("破線・網掛けはROPE [-0.1, 0.1]。事後分布の", round(pct_in_rope, 1),
                         "%がROPE内。説明用の模式図であり実データではない。scripts/R/ch15_worked_example.R より生成。"))
save_fig(p, "fig15-01-posterior-rope", chapter = "ch15", width = 6, height = 3.6)
cat("ROPE内の事後確率:", round(pct_in_rope, 1), "%\n")


## `ch15_fda_registration.R`

対象章: 第15章「不確実性と曲線をモデル化する——ベイズ統計・GAM・FDA」

「FDA——曲線を関数として扱う」節の作例。同じ形状(単一のピークを持つ
F0曲線)を持つが、ピークの生じるタイミングが話者ごとに異なる架空の
曲線群を作り、時間軸をそのまま重ねた場合(位置合わせ前)と、各曲線の
ピーク位置を共通の基準点に揃えてから重ねた場合(位置合わせ後、
landmark registration)とで、曲線群の平均がどう変わるかを比較する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_curves <- 12
t_grid   <- seq(0, 1, length.out = 200)

peak_times <- runif(n_curves, 0.35, 0.65)   # 話者ごとにピークの時刻がばらつく

make_curve <- function(peak_t) {
  100 + 30 * exp(-((t_grid - peak_t) / 0.08)^2)
}

curves <- map_dfr(seq_len(n_curves), function(i) {
  tibble(curve_id = i, t = t_grid, f0 = make_curve(peak_times[i]) + rnorm(length(t_grid), 0, 1.5))
})

# --- 位置合わせ前の平均 ---------------------------------------------------
mean_unaligned <- curves |> group_by(t) |> summarise(f0 = mean(f0), .groups = "drop")

# --- landmark registrationによる位置合わせ ------------------------------
# 各曲線のピーク時刻を、全曲線の平均ピーク時刻に揃えるよう、時間軸を
# 区分線形に伸縮する(0とt=1は固定したまま、ピーク位置だけを動かす)。
target_peak <- mean(peak_times)
warp_time <- function(t, peak_t, target) {
  ifelse(t <= peak_t,
         t / peak_t * target,
         target + (t - peak_t) / (1 - peak_t) * (1 - target))
}

curves_aligned <- curves |>
  group_by(curve_id) |>
  mutate(t_warped = warp_time(t, peak_times[curve_id[1]], target_peak)) |>
  ungroup()

# 位置合わせ後の曲線を共通グリッドに再補間してから平均する
realigned_grid <- curves_aligned |>
  group_by(curve_id) |>
  group_modify(~ tibble(t = t_grid, f0 = approx(.x$t_warped, .x$f0, xout = t_grid, rule = 2)$y)) |>
  ungroup()

mean_aligned <- realigned_grid |> group_by(t) |> summarise(f0 = mean(f0), .groups = "drop")

p1 <- ggplot() +
  geom_line(data = curves, aes(t, f0, group = curve_id), colour = "grey70", linewidth = 0.3) +
  geom_line(data = mean_unaligned, aes(t, f0), colour = "#A8681F", linewidth = 1.1) +
  labs(title = "位置合わせ前(時間軸をそのまま重ねる)", x = "正規化時間", y = "F0(模式値)") +
  theme(plot.title = element_text(size = 10))

p2 <- ggplot() +
  geom_line(data = realigned_grid, aes(t, f0, group = curve_id), colour = "grey70", linewidth = 0.3) +
  geom_line(data = mean_aligned, aes(t, f0), colour = "#34518A", linewidth = 1.1) +
  labs(title = "位置合わせ後(各曲線のピーク時刻を揃えてから重ねる)", x = "正規化時間", y = "F0(模式値)") +
  theme(plot.title = element_text(size = 10))

library(patchwork)
p <- p1 + p2
ggsave(here::here("assets", "figures", "ch15", "fig15-03-fda-registration.png"),
       p, width = 8, height = 3.6, dpi = 320)
ggsave(here::here("assets", "figures", "ch15", "fig15-03-fda-registration.pdf"),
       p, width = 8, height = 3.6, device = cairo_pdf)

cat("個々の曲線の真のピーク高: 130\n")
cat("位置合わせ前の平均曲線のピーク高:", round(max(mean_unaligned$f0), 1), "\n")
cat("位置合わせ後の平均曲線のピーク高:", round(max(mean_aligned$f0), 1), "\n")
cat("完了。図はassets/figures/ch15/に保存した。\n")


# 第16章 機械学習・深層学習・自己教師あり音声モデル

## `ch16_speaker_leakage.R`

対象章: 第16章「機械学習・深層学習・自己教師あり音声モデル」

「教師あり学習の基礎と注意点」節の作例。話者holdoutを行わない
交差検証が、真の汎化性能をどれだけ過大評価するかを、架空のデータで
実際に確認する。実測データではない。

設計: 各話者に、ラベル(たとえば無声化の有無)についての固有の生起率
(話者ごとに安定した個人差)を持たせ、音響特徴量は主に話者固有の
「声質」を反映する(ラベルとの直接の対応は弱い)ものとして生成する。
こうすると、話者を特定できる特徴量を使うモデルは、訓練データに
含まれる話者については「この話者は何%の確率でラベル1を持つか」を
記憶するだけで高い精度を出せてしまうが、その記憶は新しい話者には
使えない。話者holdoutの有無で、この記憶による見かけの精度がどれだけ
過大評価かを確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(randomForest)

set.seed(20260906)

n_speakers    <- 20
n_tok_per_spk <- 40

speakers <- tibble(
  speaker      = sprintf("s%02d", 1:n_speakers),
  # 話者固有の「声質」オフセット。分散をトークン内ノイズ(SD 1)よりずっと
  # 大きく取ることで、この2つの特徴量だけからほぼ確実に話者を再識別できる
  # ようにする(リークが起きるための必要条件)。
  speaker_re   = rnorm(n_speakers, 0, 30),
  speaker_prob = runif(n_speakers, 0.1, 0.9)     # 話者ごとに安定したラベル生起率の個人差(大きめ)
)

d <- speakers |>
  tidyr::crossing(tibble(tok = 1:n_tok_per_spk)) |>
  mutate(
    label = factor(rbinom(n(), 1, speaker_prob)),               # 話者固有の生起率に従う
    label_num = as.numeric(as.character(label)),
    feature1 = 0.15 * label_num + speaker_re + rnorm(n(), 0, 1), # ラベルへの真の寄与は弱い
    feature2 = 0.1 * label_num + speaker_re * 0.8 + rnorm(n(), 0, 1)
  ) |>
  select(speaker, label, feature1, feature2)

# --- (a) 話者を考慮しない通常の交差検証(5-fold, ランダム分割) -----------
n_folds <- 5
d$fold_random <- sample(rep(1:n_folds, length.out = nrow(d)))

acc_random <- map_dbl(1:n_folds, function(k) {
  train <- d |> filter(fold_random != k)
  test  <- d |> filter(fold_random == k)
  m <- randomForest(label ~ feature1 + feature2, data = train, ntree = 200)
  mean(predict(m, test) == test$label)
})

# --- (b) 話者holdoutによる交差検証(話者ごとにfoldを割り当て) -----------
speaker_folds <- tibble(speaker = speakers$speaker,
                          fold_speaker = sample(rep(1:n_folds, length.out = n_speakers)))
d <- d |> left_join(speaker_folds, by = "speaker")

acc_speaker_holdout <- map_dbl(1:n_folds, function(k) {
  train <- d |> filter(fold_speaker != k)
  test  <- d |> filter(fold_speaker == k)
  m <- randomForest(label ~ feature1 + feature2, data = train, ntree = 200)
  mean(predict(m, test) == test$label)
})

cat("通常の交差検証(話者リークあり)の平均精度:", round(mean(acc_random) * 100, 1), "%\n")
cat("話者holdoutによる交差検証の平均精度:", round(mean(acc_speaker_holdout) * 100, 1), "%\n")
cat("最頻クラスによるベースライン:", round(max(prop.table(table(d$label))) * 100, 1), "%\n")

results <- tibble(
  fold = rep(seq_len(n_folds), 2),
  method = rep(c("通常の交差検証\n(話者リークあり)", "話者holdout\n交差検証"), each = n_folds),
  accuracy = c(acc_random, acc_speaker_holdout)
)

p <- ggplot(results, aes(x = method, y = accuracy)) +
  geom_point(aes(group = fold), position = position_jitter(width = 0.06, height = 0),
             size = 2.2, alpha = 0.75, colour = "#0072B2") +
  stat_summary(fun = mean, geom = "crossbar", width = 0.55, linewidth = 0.65,
               colour = "#D55E00") +
  geom_hline(yintercept = max(prop.table(table(d$label))), linetype = 2, colour = "#A8681F") +
  annotate("text", x = 1.95, y = max(prop.table(table(d$label))) - 0.012,
           label = "最頻クラスベースライン", hjust = 1, vjust = 1,
           size = 3.1, colour = "#A8681F") +
  coord_cartesian(ylim = c(0.25, 0.75), clip = "off") +
  scale_y_continuous(labels = scales::percent) +
  labs(x = NULL, y = "予測精度 (各点は1分割、橙線は平均)",
       caption = "シミュレーションデータ。破線は最頻クラスベースライン。scripts/R/ch16_speaker_leakage.R より生成。")
save_fig(p, "fig16-01-speaker-holdout", chapter = "ch16", width = 6.5, height = 3)
cat("完了。図はassets/figures/ch16/に保存した。\n")
